In [ ]:
import numpy as np
import cvxpy as cp
import time
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import os

In [ ]:
def generate_2d_anatomical_sets(grid_shape, size_variation=0.2, jitter_scale=1.2, rng=None):
    """
    Generates a 2D synthetic anatomy with RANDOMLY SIZED and SHAPED organs
    for prostate cancer simulation. Creates anatomically plausible shapes:
    - PTV (prostate): Elliptical/spherical shape
    - OAR1 (Bladder): Irregular, pear-shaped or rounded
    - OAR2 (Rectum): Elongated, tubular shape
    - OAR3 (Femoral heads): Circular/elliptical shape

    Additional jitter/noise is injected to reduce duplicated structures.

    Args:
        grid_shape: Tuple (rows, cols) defining the grid size
        size_variation: Fraction of average size to vary (default 0.2 = ±20%)
        jitter_scale: Multiplier controlling how far organs can wander
        rng: Optional numpy Generator for reproducibility

    Returns:
        A dictionary of organ indices and a dict of organ 'patches' for plotting.
    """
    from scipy.ndimage import (
        binary_fill_holes,
        binary_dilation,
        binary_erosion,
        gaussian_filter,
        label,
    )

    if rng is None:
        seed = int(np.random.randint(0, 2**31 - 1))  # stay within RandomState bounds
        rng = np.random.default_rng(seed)

    m_voxels = grid_shape[0] * grid_shape[1]
    m_rows, m_cols = grid_shape

    organ_configs = {
        'ptv': (23, 23, 10, 8, 'ellipse'),
        'oar1': (20, 30, 12, 10, 'irregular'),
        'oar2': (20, 10, 14, 6, 'elongated'),
        'oar3': (34, 34, 7, 7, 'circular')
    }

    organ_sets = {}
    organ_patches = {}

    y_coords, x_coords = np.ogrid[:m_rows, :m_cols]
    jitter_span = max(4, int(5 * jitter_scale))

    for name, (center_r, center_c, avg_major, avg_minor, shape_type) in organ_configs.items():
        major_min = max(3, int(avg_major * (1 - size_variation)))
        major_max = int(avg_major * (1 + size_variation))
        minor_min = max(3, int(avg_minor * (1 - size_variation)))
        minor_max = int(avg_minor * (1 + size_variation))

        actual_major = rng.integers(major_min, major_max + 1)
        actual_minor = rng.integers(minor_min, minor_max + 1)

        r_jitter = rng.integers(-jitter_span, jitter_span + 1)
        c_jitter = rng.integers(-jitter_span, jitter_span + 1)

        center_r_new = max(actual_major // 2, min(m_rows - actual_major // 2, center_r + r_jitter))
        center_c_new = max(actual_minor // 2, min(m_cols - actual_minor // 2, center_c + c_jitter))

        mask = np.zeros(grid_shape, dtype=bool)
        angle = 0

        if shape_type in ('ellipse', 'circular'):
            angle = rng.uniform(-np.pi/3, np.pi/3)
            cos_a, sin_a = np.cos(angle), np.sin(angle)
            dx = (x_coords - center_c_new) * cos_a - (y_coords - center_r_new) * sin_a
            dy = (x_coords - center_c_new) * sin_a + (y_coords - center_r_new) * cos_a

            irregularity = rng.uniform(0.75, 1.3)
            ellipse_dist = (dx / (actual_minor / 2 * irregularity))**2 + (dy / (actual_major / 2))**2
            boundary_noise = gaussian_filter(rng.normal(0, 0.3, grid_shape), sigma=rng.uniform(0.6, 1.3))
            mask = ellipse_dist <= (1 + 0.15 * boundary_noise)

        elif shape_type == 'elongated':
            angle = rng.uniform(-np.pi/3, np.pi/3)
            cos_a, sin_a = np.cos(angle), np.sin(angle)
            dx = (x_coords - center_c_new) * cos_a - (y_coords - center_r_new) * sin_a
            dy = (x_coords - center_c_new) * sin_a + (y_coords - center_r_new) * cos_a

            elongated_major = actual_major * rng.uniform(1.0, 1.4)
            ellipse_dist = (dx / (actual_minor / 2))**2 + (dy / (elongated_major / 2))**2
            taper = 1 - 0.25 * np.abs(dy) / (elongated_major / 2)
            wiggle = 0.15 * np.sin((dy / max(1, elongated_major)) * rng.uniform(4, 7))
            mask = ellipse_dist <= (taper + wiggle)

        elif shape_type == 'irregular':
            angle = rng.uniform(-np.pi/2, np.pi/2)
            cos_a, sin_a = np.cos(angle), np.sin(angle)
            dx = (x_coords - center_c_new) * cos_a - (y_coords - center_r_new) * sin_a
            dy = (x_coords - center_c_new) * sin_a + (y_coords - center_r_new) * cos_a

            base_dist = (dx / (actual_minor / 2))**2 + (dy / (actual_major / 2))**2
            irregularity_1 = rng.uniform(0.6, 1.4)
            mask = base_dist <= irregularity_1

            for _ in range(rng.integers(2, 5)):
                offset_r = rng.integers(-4, 5)
                offset_c = rng.integers(-4, 5)
                bulge_center_r = np.clip(center_r_new + offset_r, 0, m_rows - 1)
                bulge_center_c = np.clip(center_c_new + offset_c, 0, m_cols - 1)
                bulge_sigma = rng.uniform(actual_minor / 4, actual_minor / 2)
                bulge = np.exp(-(((y_coords - bulge_center_r)**2 + (x_coords - bulge_center_c)**2) / (2 * bulge_sigma**2)))
                mask = mask | (bulge > rng.uniform(0.2, 0.35))

        mask_float = mask.astype(float)

        num_bulges = rng.integers(1, 4)
        for _ in range(num_bulges):
            bulge_center_r = rng.integers(max(0, center_r_new - actual_major), min(m_rows, center_r_new + actual_major + 1))
            bulge_center_c = rng.integers(max(0, center_c_new - actual_minor), min(m_cols, center_c_new + actual_minor + 1))
            bulge_sigma = rng.uniform(1.5, 3.5)
            bulge = np.exp(-(((y_coords - bulge_center_r)**2 + (x_coords - bulge_center_c)**2) / (2 * bulge_sigma**2)))
            mask_float += rng.uniform(0.15, 0.35) * bulge

        random_field = gaussian_filter(rng.normal(0, 1, grid_shape), sigma=rng.uniform(0.8, 1.6))
        mask_float = np.clip(mask_float + 0.2 * random_field, 0, None)
        threshold = rng.uniform(0.45, 0.6)
        mask = mask_float > threshold

        if rng.random() < 0.7:
            mask = binary_dilation(mask, iterations=rng.integers(1, 3))
        if rng.random() < 0.5:
            mask = binary_erosion(mask, iterations=1)

        mask = binary_fill_holes(mask)

        labeled_mask, num_features = label(mask)
        if num_features > 0:
            sizes = np.bincount(labeled_mask.ravel())
            if len(sizes) > 1:
                largest_component = np.argmax(sizes[1:]) + 1
                mask = (labeled_mask == largest_component)

        if np.sum(mask) < 4:
            dx = x_coords - center_c_new
            dy = y_coords - center_r_new
            mask = (dx**2 / (actual_minor / 2)**2 + dy**2 / (actual_major / 2)**2) <= 1

        organ_sets[name] = np.where(mask.ravel())[0]

        if np.sum(mask) > 0:
            rows, cols = np.where(mask)
            r_min, r_max = rows.min(), rows.max()
            c_min, c_max = cols.min(), cols.max()

            if shape_type in ('ellipse', 'circular'):
                patch = patches.Ellipse(
                    (center_c_new, center_r_new), actual_minor, actual_major,
                    angle=np.degrees(angle),
                    linewidth=1.5, edgecolor='red', facecolor='none',
                    label=f'{name} (Vol: {len(organ_sets[name])})'
                )
            else:
                patch = patches.Rectangle(
                    (c_min, r_min), (c_max - c_min), (r_max - r_min),
                    linewidth=1.5, edgecolor='red', facecolor='none',
                    label=f'{name} (Vol: {len(organ_sets[name])})'
                )
        else:
            patch = patches.Rectangle(
                (center_c_new - actual_minor // 2, center_r_new - actual_major // 2),
                actual_minor, actual_major,
                linewidth=1.5, edgecolor='red', facecolor='none',
                label=f'{name} (Vol: {len(organ_sets[name])})'
            )

        organ_patches[name] = patch

    return organ_sets, organ_patches

In [ ]:
def generate_2d_dose_influence_matrix(grid_shape, n_beamlets):
    """
    Generates a *realistic* 2D dose-influence matrix D (m x n)
    with smooth, localized 2D Gaussian falloff.
    """
    m_rows, m_cols = grid_shape
    m_voxels = m_rows * m_cols
    
    # Create an empty D matrix
    D = np.zeros((m_voxels, n_beamlets))
    
    # Create coordinate grids
    x, y = np.indices(grid_shape)
    
    for j in range(n_beamlets):
        # 1. Pick a random "center" for this beamlet
        center_x = np.random.uniform(0, m_rows)
        center_y = np.random.uniform(0, m_cols)
        
        # 2. Define the "width" (sigma) of the beam
        sigma_x = np.random.uniform(m_rows / 10, m_rows / 5)
        sigma_y = np.random.uniform(m_cols / 10, m_cols / 5)
        
        # 3. Calculate squared distance from the center for all voxels
        distance_sq = ((x - center_x)**2 / sigma_x**2 + 
                       (y - center_y)**2 / sigma_y**2)
        
        # 4. Apply Gaussian function
        dose_grid = np.exp(-distance_sq / 2)
        
        # 5. Ravel the 2D dose grid and store it as a column in D
        D[:, j] = dose_grid.ravel()
        
    return D

In [ ]:
import cvxpy as cp

def solve_forward_problem(D, organ_sets):
    """
    Solves the forward QP optimization problem.
    OAR1: Square of the sum
    OAR2: Linear sum
    OAR3: Scaled Exponential (Stable for inputs up to 160)
    """
    
    # --- 1. Define Fixed Parameters ---
    params = {
        'theta_prescribed': 78.0,
        'theta_max_PTV': 81.9,
        'theta_thresh_1': 59.15,
        'theta_thresh_2': 59.15,
        'beta_1': 0.5,
        'beta_2': 1.5,
        # Stability parameters for the Exponential term
        'exp_scale_k': 15.0,    # Normalizes input (160/15 = ~10.6)
        'alpha_exp': 1e-3       # Multiplier to keep total objective value stable
    }

    # --- 2. Get Problem Dimensions ---
    m_voxels, n_beamlets = D.shape
    
    idx_ptv = organ_sets['ptv']
    idx_oar1 = organ_sets['oar1']
    idx_oar2 = organ_sets['oar2']
    idx_oar3 = organ_sets['oar3']
    
    alpha_1 = 0.001  # Multiplier for OAR1
    alpha_2 = 1.0    # Multiplier for OAR2 (Linear)

    # --- 3. Define CVXPY Decision Variables ---
    w = cp.Variable(n_beamlets, name="w", nonneg=True)
    o_1 = cp.Variable(len(idx_oar1), name="o_1", nonneg=True)
    o_2 = cp.Variable(len(idx_oar2), name="o_2", nonneg=True)
    d_max = cp.Variable(name="d_max", nonneg=True)

    # --- 4. Define Objective Function ---
    # Term 1: Quadratic (Square of the Sum)
    obj_oar1 = alpha_1 * cp.square(cp.sum(o_1))
    
    # Term 2: Linear
    obj_oar2 = alpha_2 * cp.sum(o_2)
    
    # Term 3: Exponential (Scaled for stability)
    obj_oar3 = params['alpha_exp'] * cp.exp(d_max / params['exp_scale_k'])
    
    objective = cp.Minimize(obj_oar1 + obj_oar2 + obj_oar3)

    # --- 5. Define Constraints ---
    dose_vector = D @ w
    
    dose_ptv = dose_vector[idx_ptv]
    dose_oar1 = dose_vector[idx_oar1]
    dose_oar2 = dose_vector[idx_oar2]
    dose_oar3 = dose_vector[idx_oar3]

    constraints = []
    constraints.append(dose_ptv >= params['theta_prescribed'])
    constraints.append(dose_ptv <= params['theta_max_PTV'])
    constraints.append(o_1 >= dose_oar1 - params['theta_thresh_1'])
    constraints.append(o_2 >= dose_oar2 - params['theta_thresh_2'])
    constraints.append(d_max >= dose_oar3)
    
    sum_w = cp.sum(w)
    constraints.append(w >= (params['beta_1'] / n_beamlets) * sum_w)
    constraints.append(w <= (params['beta_2'] / n_beamlets) * sum_w)
    
    # --- 6. Solve the Problem with Cascading Fallbacks ---
    problem = cp.Problem(objective, constraints)
    mosek_failed = False
    
    try:
        # MOSEK handles exponential cones and quadratic forms efficiently
        problem.solve(solver='MOSEK', verbose=False)
    except (cp.error.SolverError, Exception):
        mosek_failed = True
        try:
            problem.solve(verbose=False) 
        except cp.error.SolverError:
            try:
                problem.solve(solver='SCS', verbose=False)
            except cp.error.SolverError:
                if len(problem.constraints) > 1:
                    del problem.constraints[1] 
                try:
                    problem.solve(solver='SCS', verbose=False) 
                except cp.error.SolverError:
                    return None, None, "Relaxed_Solve_Failed", mosek_failed

    # --- 7. Return Results ---
    if problem.status not in ["optimal", "optimal_inaccurate"]:
        return None, None, problem.status, mosek_failed
    
    return w.value, problem.value, problem.status, mosek_failed

In [ ]:
import os
import time
import numpy as np
import hashlib
import matplotlib.patches as patches
import cvxpy as cp

# --- IMPORTS FROM YOUR SOLVER/GENERATOR ---
# You must ensure these functions are available in your environment.
# If they are in a separate file (e.g., 'my_simulation.py'), import them:
# from my_simulation import generate_2d_anatomical_sets, generate_2d_dose_influence_matrix, solve_forward_problem

# --- Configuration ---
N_PATIENTS = 30000     # Total iterations
GRID_SHAPE = (55, 55)     
N_BEAMLETS = 100          
MIN_VOXEL_COUNT = 10      # Hard limit: organ too small
MAX_OVERLAP_PCT = 50.0    # Hard limit: PTV overlapping OAR too much
GLOBAL_SEED = 142         

# Output Directory
DATA_DIR = "./worst_case_search"
os.makedirs(DATA_DIR, exist_ok=True)

# --- Validation Function (Adapted for Streaming) ---
def is_anatomically_valid(organ_data, organ_names=['ptv', 'oar1', 'oar2', 'oar3']):
    """
    Checks if the generated anatomy is biologically possible.
    Returns: (bool, reason)
    """
    ptv_indices = np.array(organ_data['ptv'], dtype=np.int32)
    ptv_set = set(ptv_indices.tolist())
    
    # 1. EMPTY OR MICRO-ORGAN CHECK
    for organ in organ_names:
        voxels = organ_data[organ]
        size = len(voxels)
        
        if size == 0:
            return False, f"Empty {organ}"
        if size < MIN_VOXEL_COUNT:
            return False, f"{organ} too small ({size} voxels)"

    # 2. OVERLAP CHECKS (PTV vs OARs)
    # We skip the specific OAR1 vs PTV size check here to allow for 
    # worst-case scenarios where OARs might be weirdly small, 
    # provided they aren't micro-noise (<10 voxels).
    
    if len(ptv_set) > 0:
        for oar in ['oar1', 'oar2', 'oar3']:
            oar_set = set(organ_data[oar])
            overlap_voxels = len(ptv_set & oar_set)
            
            # Avoid division by zero
            if len(ptv_set) == 0: continue

            overlap_pct = (overlap_voxels / len(ptv_set)) * 100.0
            
            if overlap_pct > MAX_OVERLAP_PCT:
                return False, f"Impossible Overlap: PTV intersects {oar} by {overlap_pct:.1f}%"

    return True, "Valid"

# --- Main Execution ---
if __name__ == "__main__":
    print(f"--- Starting Worst-Case Search (N={N_PATIENTS}) ---")
    print(f"--- Output Directory: {DATA_DIR} ---")

    np.random.seed(GLOBAL_SEED)
    
    # Tracking
    max_obj_val = -float('inf')
    best_patient_id = -1
    
    count_valid_solved = 0
    count_rejected_anatomy = 0
    count_solver_failed = 0
    
    start_time = time.time()

    for i in range(N_PATIENTS):
        if i % 100 == 0:
            elapsed = time.time() - start_time
            print(f"Iter {i} | Valid: {count_valid_solved} | Rejected: {count_rejected_anatomy} | Current Max: {max_obj_val:.4f} | Time: {elapsed:.1f}s")

        # -------------------------------------------------
        # 1. GENERATE
        # -------------------------------------------------
        # (Assuming these functions exist in your scope)
        organ_sets, organ_patches = generate_2d_anatomical_sets(GRID_SHAPE)
        D = generate_2d_dose_influence_matrix(GRID_SHAPE, N_BEAMLETS)

        # -------------------------------------------------
        # 2. VALIDATE (The Filter)
        # -------------------------------------------------
        # We check validity BEFORE solving to save time
        is_valid, reason = is_anatomically_valid(organ_sets)
        
        if not is_valid:
            count_rejected_anatomy += 1
            # Skip the expensive solver!
            continue

        # -------------------------------------------------
        # 3. SOLVE
        # -------------------------------------------------
        # Only solve if anatomy is valid
        w_opt, obj_val, status, mosek_failed = solve_forward_problem(D, organ_sets)

        if mosek_failed or status not in ["optimal", "optimal_inaccurate"]:
            count_solver_failed += 1
            continue

        count_valid_solved += 1

        # -------------------------------------------------
        # 4. CHECK FOR WORST CASE (MAX PENALTY)
        # -------------------------------------------------
        if obj_val > max_obj_val:
            print(f"\n[!] New Max Penalty found at Iteration {i}")
            print(f"    Old Max: {max_obj_val:.4f} -> New Max: {obj_val:.4f}")
            print(f"    Reason for update: Valid anatomy with higher objective.")

            max_obj_val = obj_val
            best_patient_id = i

            # Save immediately
            np.save(f'{DATA_DIR}/D_worst_case.npy', D)
            np.save(f'{DATA_DIR}/organ_sets_worst_case.npy', organ_sets, allow_pickle=True)
            np.save(f'{DATA_DIR}/w_opt_worst_case.npy', w_opt)
            
            # Save metadata
            with open(f'{DATA_DIR}/worst_case_info.txt', 'w') as f:
                f.write(f"Iteration ID: {i}\n")
                f.write(f"Objective Value: {obj_val}\n")
                f.write(f"Solver Status: {status}\n")

    # --- Final Report ---
    total_time = time.time() - start_time
    print("\n" + "="*40)
    print("SEARCH COMPLETE")
    print("="*40)
    print(f"Total Iterations:     {N_PATIENTS}")
    print(f"Anatomy Rejections:   {count_rejected_anatomy} ({(count_rejected_anatomy/N_PATIENTS)*100:.1f}%)")
    print(f"Solver Failures:      {count_solver_failed}")
    print(f"Valid Solutions:      {count_valid_solved}")
    print(f"Time Taken:           {total_time/60:.2f} minutes")
    print("-" * 40)
    print(f"HIGHEST OBJECTIVE:    {max_obj_val:.6f}")
    print(f"FOUND AT ITERATION:   {best_patient_id}")
    print("="*40)

In [ ]:
import os
import time
import numpy as np
import hashlib
import matplotlib.patches as patches
import cvxpy as cp

# --- IMPORTS FROM YOUR SOLVER/GENERATOR ---
# You must ensure these functions are available in your environment.
# from my_simulation import generate_2d_anatomical_sets, generate_2d_dose_influence_matrix, solve_forward_problem

# --- Configuration ---
N_PATIENTS = 30000     # Total iterations
GRID_SHAPE = (55, 55)     
N_BEAMLETS = 100          
MIN_VOXEL_COUNT = 10      # Hard limit: organ too small
MAX_OVERLAP_PCT = 50.0    # Hard limit: PTV overlapping OAR too much
GLOBAL_SEED = 142         

# Output Directory
DATA_DIR = "./best_case_search"
os.makedirs(DATA_DIR, exist_ok=True)

# --- Validation Function ---
def is_anatomically_valid(organ_data, organ_names=['ptv', 'oar1', 'oar2', 'oar3']):
    """
    Checks if the generated anatomy is biologically possible.
    Returns: (bool, reason)
    """
    ptv_indices = np.array(organ_data['ptv'], dtype=np.int32)
    ptv_set = set(ptv_indices.tolist())
    
    # 1. EMPTY OR MICRO-ORGAN CHECK
    for organ in organ_names:
        voxels = organ_data[organ]
        size = len(voxels)
        
        if size == 0:
            return False, f"Empty {organ}"
        if size < MIN_VOXEL_COUNT:
            return False, f"{organ} too small ({size} voxels)"

    # 2. OVERLAP CHECKS (PTV vs OARs)
    if len(ptv_set) > 0:
        for oar in ['oar1', 'oar2', 'oar3']:
            oar_set = set(organ_data[oar])
            overlap_voxels = len(ptv_set & oar_set)
            
            # Avoid division by zero
            if len(ptv_set) == 0: continue

            overlap_pct = (overlap_voxels / len(ptv_set)) * 100.0
            
            if overlap_pct > MAX_OVERLAP_PCT:
                return False, f"Impossible Overlap: PTV intersects {oar} by {overlap_pct:.1f}%"

    return True, "Valid"

# --- Main Execution ---
if __name__ == "__main__":
    print(f"--- Starting Best-Case Search (N={N_PATIENTS}) ---")
    print(f"--- Output Directory: {DATA_DIR} ---")

    np.random.seed(GLOBAL_SEED)
    
    # Tracking for BEST CASE (Minimum Objective)
    # We initialize with infinity so the first valid solution will be lower
    min_obj_val = float('inf') 
    best_patient_id = -1
    
    count_valid_solved = 0
    count_rejected_anatomy = 0
    count_solver_failed = 0
    
    start_time = time.time()

    for i in range(N_PATIENTS):
        if i % 100 == 0:
            elapsed = time.time() - start_time
            # Log 'Current Min'
            current_display = min_obj_val if min_obj_val != float('inf') else 0.0
            print(f"Iter {i} | Valid: {count_valid_solved} | Rejected: {count_rejected_anatomy} | Current Min: {current_display:.4f} | Time: {elapsed:.1f}s")

        # -------------------------------------------------
        # 1. GENERATE
        # -------------------------------------------------
        organ_sets, organ_patches = generate_2d_anatomical_sets(GRID_SHAPE)
        D = generate_2d_dose_influence_matrix(GRID_SHAPE, N_BEAMLETS)

        # -------------------------------------------------
        # 2. VALIDATE
        # -------------------------------------------------
        is_valid, reason = is_anatomically_valid(organ_sets)
        
        if not is_valid:
            count_rejected_anatomy += 1
            continue

        # -------------------------------------------------
        # 3. SOLVE
        # -------------------------------------------------
        w_opt, obj_val, status, mosek_failed = solve_forward_problem(D, organ_sets)

        if mosek_failed or status not in ["optimal", "optimal_inaccurate"]:
            count_solver_failed += 1
            continue

        count_valid_solved += 1

        # -------------------------------------------------
        # 4. CHECK FOR BEST CASE (MIN PENALTY)
        # -------------------------------------------------
        # We look for the LOWEST objective value (easiest case to treat)
        if obj_val < min_obj_val:
            print(f"\n[!] New Best Case (Min Penalty) found at Iteration {i}")
            print(f"    Old Min: {min_obj_val:.4f} -> New Min: {obj_val:.4f}")
            print(f"    Reason for update: Valid anatomy with lower objective.")

            min_obj_val = obj_val
            best_patient_id = i

            # Save immediately
            np.save(f'{DATA_DIR}/D_best_case.npy', D)
            np.save(f'{DATA_DIR}/organ_sets_best_case.npy', organ_sets, allow_pickle=True)
            np.save(f'{DATA_DIR}/w_opt_best_case.npy', w_opt)
            
            # Save metadata
            with open(f'{DATA_DIR}/best_case_info.txt', 'w') as f:
                f.write(f"Iteration ID: {i}\n")
                f.write(f"Objective Value: {obj_val}\n")
                f.write(f"Solver Status: {status}\n")

    # --- Final Report ---
    total_time = time.time() - start_time
    print("\n" + "="*40)
    print("SEARCH COMPLETE (BEST CASE)")
    print("="*40)
    print(f"Total Iterations:     {N_PATIENTS}")
    print(f"Anatomy Rejections:   {count_rejected_anatomy} ({(count_rejected_anatomy/N_PATIENTS)*100:.1f}%)")
    print(f"Solver Failures:      {count_solver_failed}")
    print(f"Valid Solutions:      {count_valid_solved}")
    print(f"Time Taken:           {total_time/60:.2f} minutes")
    print("-" * 40)
    print(f"LOWEST OBJECTIVE:     {min_obj_val:.6f}")
    print(f"FOUND AT ITERATION:   {best_patient_id}")

In [ ]:
import os
import time
import numpy as np
# Assuming other necessary imports (organ_sets logic, solver logic) are present in your environment

# --- 1. Simulation Setup ---
N_PATIENTS = 2000      # Set number of patients
GRID_SHAPE = (55, 55)  # Grid dimensions
N_BEAMLETS = 100       # 100 beamlets
M_VOXELS = GRID_SHAPE[0] * GRID_SHAPE[1]
GLOBAL_SEED = 142    # Single master seed

# Create a directory to store patient data
DATA_DIR = "./patient_data_high_std"
os.makedirs(DATA_DIR, exist_ok=True)
print(f"Saving all data to: {DATA_DIR}")

# --- 2. Data Storage Initialization ---
all_objective_values = []
mosek_failed_patients = []
total_failed_patients = []
successful_patients = 0

# --- 3. Start Generation Loop ---
print(f"\n--- Starting Generation of {N_PATIENTS} Patients ---")
print(f"--- Using Global Seed: {GLOBAL_SEED} ---")

# Initialize seed once before the loop
np.random.seed(GLOBAL_SEED) 

start_time_total = time.time()

for i in range(N_PATIENTS):
    print(f"Generating Patient {i}...")
    
    
    
    organ_sets, organ_patches = generate_2d_anatomical_sets(GRID_SHAPE)
    D = generate_2d_dose_influence_matrix(GRID_SHAPE, N_BEAMLETS)
    
    # 2. Solve the problem
    w_opt, obj_val, status, mosek_failed = solve_forward_problem(D, organ_sets)
    
    # Track MOSEK-specific failures
    if mosek_failed:
        mosek_failed_patients.append(i)
    
    # 3. Check for success and save data
    if status in ["optimal", "optimal_inaccurate"]:
        # Save D matrix
        np.save(f'{DATA_DIR}/D_patient_{i}.npy', D)
        
        # Save organ_sets
        np.save(f'{DATA_DIR}/organ_sets_patient_{i}.npy', organ_sets, allow_pickle=True)
        
        # Save organ_patches
        np.save(f'{DATA_DIR}/organ_patches_patient_{i}.npy', organ_patches, allow_pickle=True)
        
        # Save w_opt individually
        np.save(f'{DATA_DIR}/w_opt_patient_{i}.npy', w_opt)
        
        # Append objective to our list
        all_objective_values.append(obj_val)
        successful_patients += 1
        
        print(f"  > Patient {i} solved. Status: {status}. Obj: {obj_val:.2f}")
        
    else:
        total_failed_patients.append(i)
        print(f"  > ! Patient {i} FAILED to solve. Status: {status}. Skipping.")

# --- 4. Save Aggregated Objective Results ---
np.save(f'{DATA_DIR}/all_objective_values.npy', np.array(all_objective_values))

end_time_total = time.time()
print("\n--- Generation Complete ---")
print(f"Total time: {(end_time_total - start_time_total) / 60:.2f} minutes.")
print(f"Data saved to '{DATA_DIR}' directory.")

# --- 5. Failure Report ---
print("\n--- Solver Failure Report ---")
print(f"Successfully generated: {successful_patients} / {N_PATIENTS} patients.")
print(f"Total failures:       {len(total_failed_patients)} / {N_PATIENTS} patients.")
print(f"MOSEK solver failures:  {len(mosek_failed_patients)} / {N_PATIENTS} patients.")

if len(total_failed_patients) > 0:
    print(f"\nList of ALL failed patient IDs:")
    print(total_failed_patients)

if len(mosek_failed_patients) > 0:
    print(f"\nList of patient IDs where MOSEK failed (may have been solved by fallback):")
    print(mosek_failed_patients)


In [ ]:
import numpy as np
import os

# --- 1. Configuration (must match your main script) ---
N_PATIENTS = N_PATIENTS     # The total number of patients you *tried* to generate
GRID_SHAPE = (55, 55)      # MUST match the grid shape used during generation
N_BEAMLETS = 100            # MUST match the number of beamlets used during generation
M_VOXELS = GRID_SHAPE[0] * GRID_SHAPE[1]  # Expected number of voxels
EXPECTED_D_SHAPE = (M_VOXELS, N_BEAMLETS)  # Expected D matrix shape
EXPECTED_W_SHAPE = (N_BEAMLETS,)           # Expected w vector shape

DATA_DIR = "./patient_data_high_std"
print(f"Loading data from: {DATA_DIR}")
print(f"Expected grid shape: {GRID_SHAPE} ({M_VOXELS} voxels)")
print(f"Expected D matrix shape: {EXPECTED_D_SHAPE}")
print(f"Expected w vector shape: {EXPECTED_W_SHAPE}")

# --- 2. Initialize lists to hold successful data ---
all_successful_D = []
all_successful_w = []
all_successful_organ_sets = []
successful_patient_ids = []
skipped_patients = []  # Track patients with mismatched shapes

# --- 3. Loop and gather all successful patient data ---
print(f"\nScanning for successful patient data (up to {N_PATIENTS})...")

for i in range(N_PATIENTS):
    # Define the filepaths for this patient
    w_file = f'{DATA_DIR}/w_opt_patient_{i}.npy'
    d_file = f'{DATA_DIR}/D_patient_{i}.npy'
    organs_file = f'{DATA_DIR}/organ_sets_patient_{i}.npy'
    
    # Check if the core data (w and D) exists.
    # This is our check for a "successful" patient.
    if os.path.exists(w_file) and os.path.exists(d_file):
        try:
            # Load the data first to check shapes
            D_loaded = np.load(d_file)
            w_loaded = np.load(w_file)
            
            # --- CRITICAL: Check if shapes match expected grid size ---
            if D_loaded.shape != EXPECTED_D_SHAPE:
                print(f"  > Skipping Patient {i}: D matrix shape {D_loaded.shape} doesn't match expected {EXPECTED_D_SHAPE}")
                skipped_patients.append((i, D_loaded.shape))
                continue
                
            if w_loaded.shape != EXPECTED_W_SHAPE:
                print(f"  > Skipping Patient {i}: w vector shape {w_loaded.shape} doesn't match expected {EXPECTED_W_SHAPE}")
                skipped_patients.append((i, w_loaded.shape))
                continue
            
            # Shapes match! Add to our lists
            all_successful_w.append(w_loaded)
            all_successful_D.append(D_loaded)
            all_successful_organ_sets.append(
                np.load(organs_file, allow_pickle=True).item()
            )
            
            # Keep track of which patient ID this was
            successful_patient_ids.append(i)
            
        except Exception as e:
            print(f"  > Warning: Found files for Patient {i}, but failed to load. Error: {e}")
            
# --- 4. Report skipped patients ---
if skipped_patients:
    print(f"\n  > Skipped {len(skipped_patients)} patients with mismatched shapes (likely from different grid size)")
    print(f"  > These patients were generated with a different GRID_SHAPE and will be excluded")
            
# --- 5. Consolidate and Save Aggregated Data ---
print("\nConsolidating all successful data...")

if successful_patient_ids:
    # --- D and W Matrices ---
    # Stack all D matrices into a 3D array: (n_patients, m_voxels, n_beamlets)
    # Stack all w vectors into a 2D array: (n_patients, n_beamlets)
    
    D_aggregated = np.stack(all_successful_D, axis=0)
    w_aggregated = np.stack(all_successful_w, axis=0)
    
    # --- Objective Values ---
    # Load the original objectives file and filter to match successful patients
    obj_original_all = np.load(f'{DATA_DIR}/all_objective_values.npy')
    
    # --- Filter objectives to only include successful patients ---
    # The objectives file contains values for ALL patients ever generated,
    # but we only want objectives for patients that passed the shape check
    # Filter to only include patient IDs that are within bounds
    valid_ids = [pid for pid in successful_patient_ids if pid < len(obj_original_all)]
    
    if len(valid_ids) != len(successful_patient_ids):
        print(f"  > WARNING: {len(successful_patient_ids) - len(valid_ids)} patient IDs are out of bounds for objectives array.")
        print(f"  > Using only {len(valid_ids)} valid patient IDs.")
        # Update successful_patient_ids and corresponding arrays
        valid_indices = [i for i, pid in enumerate(successful_patient_ids) if pid in valid_ids]
        successful_patient_ids = [successful_patient_ids[i] for i in valid_indices]
        D_aggregated = D_aggregated[valid_indices]
        w_aggregated = w_aggregated[valid_indices]
        # all_successful_organ_sets is a list, so use list comprehension
        all_successful_organ_sets = [all_successful_organ_sets[i] for i in valid_indices]
    
    obj_original = obj_original_all[valid_ids]

    # --- Verification ---
    n_successful = len(successful_patient_ids)
    print(f"  > Found {n_successful} successful patients.")
    print(f"  > Shape of aggregated D: {D_aggregated.shape}")
    print(f"  > Shape of aggregated w: {w_aggregated.shape}")
    
    if len(obj_original) == n_successful:
        print(f"  > Objective values count ({len(obj_original)}) matches. This is good.")
    else:
        print(f"  > WARNING: Mismatch in objective values!")
        print(f"  > Found {n_successful} w/D files but {len(obj_original)} objectives.")
        print(f"  > Original objectives file had {len(obj_original_all)} values.")
        print(f"  > Filtered to {len(obj_original)} values matching successful patient IDs.")

    # --- 5. Save the new, clean files ---
    CLEAN_DIR = "./patient_data_clean_high_std"
    os.makedirs(CLEAN_DIR, exist_ok=True)
    print(f"\nSaving new aggregated files to: {CLEAN_DIR}")
    
    # Save the big D and w arrays
    np.save(f'{CLEAN_DIR}/D_all_successful.npy', D_aggregated)
    np.save(f'{CLEAN_DIR}/w_all_successful.npy', w_aggregated)
    
    # Save the objectives, aligned with the D/w arrays (filtered to successful patients only)
    np.save(f'{CLEAN_DIR}/objectives_all_successful.npy', obj_original)
    
    # Save the list of *original* patient IDs for your records
    np.save(f'{CLEAN_DIR}/successful_original_ids.npy', np.array(successful_patient_ids))
    
    # Save the organ sets (this will be an array of dicts)
    np.save(f'{CLEAN_DIR}/organ_sets_all_successful.npy', 
            all_successful_organ_sets, allow_pickle=True)
    
    print("--- Aggregation Complete ---")

else:
    print("No successful patient data was found to aggregate.")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import hashlib
from sklearn.model_selection import train_test_split

# --- Configuration & Globals ---
GRID_SHAPE = (55, 55) 
M_VOXELS = GRID_SHAPE[0] * GRID_SHAPE[1]

# Cleaning Thresholds
STD_DEV_THRESHOLD = 3.0       # Z-score limit for outliers
MIN_VOXEL_COUNT = 10          # Biological hard limit
MAX_OVERLAP_PCT = 50.0        # PTV cannot overlap >50% with healthy organs (Physiological limit)

# Paths
CLEAN_DIR = "./patient_data_clean_high_std"  # Input directory
ML_DIR = "./ml_data_high_std"                # Output directory
TEST_SET_SIZE = 200
RANDOM_STATE = 42

# Ensure output directory exists
os.makedirs(ML_DIR, exist_ok=True)

def load_full_dataset():
    """
    Loads the complete dataset: D, w, organ_sets, objectives, and IDs.
    """
    print(f"--- Loading data from: {CLEAN_DIR} ---")
    try:
        data = {}
        data['D'] = np.load(f'{CLEAN_DIR}/D_all_successful.npy')
        data['w'] = np.load(f'{CLEAN_DIR}/w_all_successful.npy')
        data['organs'] = np.load(f'{CLEAN_DIR}/organ_sets_all_successful.npy', allow_pickle=True)
        data['obj'] = np.load(f'{CLEAN_DIR}/objectives_all_successful.npy')
        data['ids'] = np.load(f'{CLEAN_DIR}/successful_original_ids.npy')
        
        # Verify integrity
        n_samples = data['D'].shape[0]
        assert n_samples == data['w'].shape[0]
        assert n_samples == len(data['organs'])
        assert n_samples == data['obj'].shape[0]
        assert n_samples == data['ids'].shape[0]
        
        print(f"Successfully loaded {n_samples} patient samples.")
        return data
        
    except FileNotFoundError as e:
        print(f"\nError: Could not find data file: {e.filename}")
        print("Please run the aggregation script first.")
        return None
    except AssertionError:
        print("\nError: Data arrays have mismatched lengths.")
        return None

def get_population_stats(organ_sets, organ_names):
    """
    Calculates the Mean and Std Dev for the CURRENT population.
    """
    stats = {}
    for organ in organ_names:
        sizes = [len(p[organ]) for p in organ_sets]
        stats[organ] = {
            'mean': np.mean(sizes),
            'std': np.std(sizes)
        }
    return stats

def check_patient_validity(patient, stats, organ_names, known_hashes):
    """
    Returns (True, "Reason") if valid, (False, "Reason") if invalid.
    """
    ptv_indices = np.array(patient['ptv'], dtype=np.int32)
    ptv_set = set(ptv_indices.tolist())
    
    # 1. DUPLICATE CHECK (Anatomy Hash)
    # We create a unique signature based on the PTV shape/position
    ptv_sig = hashlib.sha1(np.sort(ptv_indices).tobytes()).hexdigest()
    if ptv_sig in known_hashes:
        return False, "Duplicate Anatomy (Exact Copy)"
    # Add to known hashes (passed by reference, so this updates the tracker)
    known_hashes.add(ptv_sig)

    # 2. BIOLOGICAL HARD LIMITS (Empty or Micro-organs)
    for organ in organ_names:
        voxels = patient[organ]
        size = len(voxels)
        
        if size == 0:
            return False, f"Empty {organ}"
        
        if size < MIN_VOXEL_COUNT:
            return False, f"{organ} too small (biological noise): {size} voxels"

    # 3. STATISTICAL OUTLIERS (The "3 Sigma" Rule)
    for organ in organ_names:
        size = len(patient[organ])
        mean = stats[organ]['mean']
        std = stats[organ]['std']
        
        # Calculate Z-score
        z_score = abs(size - mean) / (std + 1e-6)
        
        if z_score > STD_DEV_THRESHOLD:
            return False, f"{organ} statistical outlier (Z={z_score:.1f})"

    # 4. OVERLAP & GEOMETRY CHECKS (Medical Logic)
    # PTV vs OARs
    for oar in ['oar1', 'oar2', 'oar3']:
        oar_set = set(patient[oar])
        if len(ptv_set) > 0:
            overlap_voxels = len(ptv_set & oar_set)
            overlap_pct = (overlap_voxels / len(ptv_set)) * 100.0
            
            if overlap_pct > MAX_OVERLAP_PCT:
                return False, f"Impossible Overlap: PTV intersects {oar} by {overlap_pct:.1f}%"

    # Relative Size: Bladder (OAR1) should generally be larger than Prostate (PTV)
    if len(patient['oar1']) < len(patient['ptv']):
        # Soft check: only fail if drastic (e.g., Bladder is < 50% of Prostate)
        if len(patient['oar1']) < 0.5 * len(patient['ptv']):
             return False, "Anatomy Error: Bladder (OAR1) significantly smaller than Prostate"

    return True, "Valid"

def clean_dataset(data):
    """
    Filters ALL data arrays (D, w, organs, obj, ids) based on organ validity.
    """
    organ_sets = data['organs']
    organ_names = ['ptv', 'oar1', 'oar2', 'oar3']
    
    # Calculate Baseline Statistics
    baseline_stats = get_population_stats(organ_sets, organ_names)
    
    valid_indices = []
    rejection_reasons = Counter()
    known_hashes = set() # To track duplicates
    
    print(f"\n--- Starting Filter on {len(organ_sets)} Patients ---")
    
    for i, patient in enumerate(organ_sets):
        # We pass known_hashes to detect duplicates on the fly
        is_valid, reason = check_patient_validity(patient, baseline_stats, organ_names, known_hashes)
        
        if is_valid:
            valid_indices.append(i)
        else:
            rejection_reasons[reason] += 1
            if sum(rejection_reasons.values()) < 5:
                print(f"  Dropped Patient Index {i} (ID: {data['ids'][i]}): {reason}")

    # Convert to numpy array for indexing
    idx = np.array(valid_indices)
    
    cleaned_data = {
        'D': data['D'][idx],
        'w': data['w'][idx],
        'organs': data['organs'][idx],
        'obj': data['obj'][idx],
        'ids': data['ids'][idx]
    }

    print("\n--- Cleaning Complete ---")
    print(f"Original Count: {len(organ_sets)}")
    print(f"Final Count:    {len(cleaned_data['organs'])}")
    print(f"Dropped:        {len(organ_sets) - len(cleaned_data['organs'])}")
    print("\nRejection Reasons:")
    for reason, count in rejection_reasons.most_common():
        print(f"  - {reason}: {count}")
        
    return cleaned_data

def perform_audit(organ_sets, title_prefix="Cleaned Data"):
    """
    Performs a statistical audit and generates plots for the provided organ sets.
    """
    print(f"\n--- Randomness Audit: {title_prefix} ---")
    
    n_patients = len(organ_sets)
    if n_patients == 0:
        print("No patients to audit.")
        return

    organ_names = ['ptv', 'oar1', 'oar2', 'oar3']
    color_map = {'ptv': 'tab:red', 'oar1': 'tab:blue', 'oar2': 'tab:orange', 'oar3': 'tab:green'}

    size_records = {name: [] for name in organ_names}
    occupancy_maps = {name: np.zeros(M_VOXELS, dtype=np.int32) for name in organ_names}
    duplicate_trackers = {name: Counter() for name in organ_names}
    overlap_records = {name: [] for name in ['oar1', 'oar2', 'oar3']}

    for patient_data in organ_sets:
        ptv_indices = np.array(patient_data['ptv'], dtype=np.int32)
        ptv_set = set(ptv_indices.tolist())

        for organ in organ_names:
            indices = np.array(patient_data[organ], dtype=np.int32)
            size_records[organ].append(len(indices))
            if len(indices) > 0:
                occupancy_maps[organ][indices] += 1
                signature = hashlib.sha1(np.sort(indices).tobytes()).hexdigest()
                duplicate_trackers[organ][signature] += 1

        for organ in ['oar1', 'oar2', 'oar3']:
            oar_set = set(np.array(patient_data[organ], dtype=np.int32).tolist())
            overlap_pct = 100.0 * len(ptv_set & oar_set) / max(1, len(ptv_set))
            overlap_records[organ].append(overlap_pct)

    # --- Print summary stats ---
    print("\nVoxel-count distributions (voxels per structure):")
    for organ in organ_names:
        sizes = np.array(size_records[organ])
        if len(sizes) == 0:
            continue
        coverage_pct = 100.0 * np.count_nonzero(occupancy_maps[organ]) / M_VOXELS
        duplicate_hits = sum(count for count in duplicate_trackers[organ].values() if count > 1)
        print(f"  {organ.upper():<4}: mean={sizes.mean():5.1f}, std={sizes.std():5.1f}, min={sizes.min():4d}, max={sizes.max():4d}, coverage={coverage_pct:4.1f}% of grid, exact duplicates={duplicate_hits}")

    print("\nPTV overlap with OARs (percent of PTV volume shared):")
    for organ in ['oar1', 'oar2', 'oar3']:
        overlaps = np.array(overlap_records[organ])
        print(f"  PTV ∩ {organ.upper()}: mean={overlaps.mean():4.2f}%, std={overlaps.std():4.2f}%, max={overlaps.max():4.2f}%")

    # --- Visualization: voxel-count histograms ---
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes = axes.ravel()
    for ax, organ in zip(axes, organ_names):
        ax.hist(size_records[organ], bins=25, color=color_map[organ], alpha=0.75)
        ax.set_title(f"{organ.upper()} voxel counts")
        ax.set_xlabel('Voxel count')
        ax.set_ylabel('Patients')
        ax.grid(True, linestyle='--', alpha=0.3)
    fig.suptitle(f'Distribution of structure sizes ({title_prefix})', fontsize=14)
    fig.tight_layout()

    # --- Visualization: overlap histograms ---
    fig2, axes2 = plt.subplots(1, 3, figsize=(15, 4))
    for ax, organ in zip(axes2, ['oar1', 'oar2', 'oar3']):
        ax.hist(overlap_records[organ], bins=20, color=color_map[organ], alpha=0.75)
        ax.set_title(f"PTV overlap with {organ.upper()}")
        ax.set_xlabel('% of PTV voxels shared')
        ax.set_ylabel('Patients')
        ax.grid(True, linestyle='--', alpha=0.3)
    fig2.suptitle(f'Tumor vs OAR overlap percentages ({title_prefix})', fontsize=14)
    fig2.tight_layout()

    # --- Visualization: occupancy heatmaps ---
    fig3, axes3 = plt.subplots(1, 4, figsize=(18, 4))
    for ax, organ in zip(axes3, organ_names):
        occupancy_ratio = occupancy_maps[organ].reshape(GRID_SHAPE) / n_patients
        im = ax.imshow(occupancy_ratio, cmap='magma', vmin=0, vmax=occupancy_ratio.max())
        ax.set_title(f"{organ.upper()} occupancy")
        ax.axis('off')
        fig3.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Fraction')
    fig3.suptitle(f'Spatial usage frequency ({title_prefix})', fontsize=14)
    fig3.tight_layout()
    
    plt.show()

def save_splits(train_data, test_data, directory):
    """
    Saves the split dictionaries to the output directory.
    """
    # Training Files
    np.save(f'{directory}/D_train.npy', train_data['D'])
    np.save(f'{directory}/w_train.npy', train_data['w'])
    np.save(f'{directory}/organ_sets_train.npy', train_data['organs'], allow_pickle=True)
    np.save(f'{directory}/objectives_train.npy', train_data['obj'])
    np.save(f'{directory}/original_ids_train.npy', train_data['ids'])

    # Testing Files
    np.save(f'{directory}/D_test.npy', test_data['D'])
    np.save(f'{directory}/w_test.npy', test_data['w'])
    np.save(f'{directory}/organ_sets_test.npy', test_data['organs'], allow_pickle=True)
    np.save(f'{directory}/objectives_test.npy', test_data['obj'])
    np.save(f'{directory}/original_ids_test.npy', test_data['ids'])
    
    print(f"\nSaved partitioned data to: {directory}")

# --- Main Execution ---
if __name__ == "__main__":
    
    # 1. Load Full Dataset
    raw_data = load_full_dataset()
    
    if raw_data is not None:
        # 2. Clean Dataset (Filter by biology/stats)
        clean_data = clean_dataset(raw_data)
        
        # 3. Audit Cleaned Data
        perform_audit(clean_data['organs'], title_prefix="Cleaned Validated Data")
        
        if len(clean_data['organs']) > TEST_SET_SIZE:
            # 4. Partition (Train / Test Split)
            print(f"\nSplitting data into {TEST_SET_SIZE} test samples...")
            
            # We unpack the dictionary to list for splitting, then repack
            arrays = [clean_data['D'], clean_data['w'], clean_data['organs'], clean_data['obj'], clean_data['ids']]
            
            splits = train_test_split(
                *arrays,
                test_size=TEST_SET_SIZE,
                random_state=RANDOM_STATE,
                shuffle=True
            )
            
            # Repackaging into dictionaries for easier saving
            # splits structure: [D_train, D_test, w_train, w_test, ...]
            train_dict = {
                'D': splits[0], 'w': splits[2], 'organs': splits[4], 
                'obj': splits[6], 'ids': splits[8]
            }
            test_dict = {
                'D': splits[1], 'w': splits[3], 'organs': splits[5], 
                'obj': splits[7], 'ids': splits[9]
            }

            print(f"Training samples: {len(train_dict['ids'])}")
            print(f"Testing samples:  {len(test_dict['ids'])}")

            # 5. Save
            save_splits(train_dict, test_dict, ML_DIR)
            
        else:
            print("CRITICAL: Not enough valid patients remaining to form a test set!")

In [ ]:
import numpy as np
import os

# --- 1. Configuration ---
ML_DIR = "./ml_data_high_std"           # Where your training data lives
WORST_CASE_DIR = "./worst_case_search"  # Directory where worst case search saved files
BEST_CASE_DIR = "./best_case_search"    # Directory where best case search saved files

# Fixed parameters for Objective Calculation
PARAMS = {
    'theta_prescribed': 78.0,
    'theta_max_PTV': 81.9,
    'theta_thresh_1': 59.15,
    'theta_thresh_2': 59.15,
    'beta_1': 0.5,
    'beta_2': 1.5,
    'exp_scale_k': 15.0,    # Scaling for stability in exponent
    'alpha_exp': 1e-3       # 1e-3 multiplier
}

print(f"--- Augmenting Training Set in {ML_DIR} ---")

# --- 2. Load Existing Training Data ---
try:
    D_train = np.load(f'{ML_DIR}/D_train.npy')
    w_train = np.load(f'{ML_DIR}/w_train.npy')
    organ_sets_train = np.load(f'{ML_DIR}/organ_sets_train.npy', allow_pickle=True)
    objectives_train = np.load(f'{ML_DIR}/objectives_train.npy')
    original_ids_train = np.load(f'{ML_DIR}/original_ids_train.npy')
except FileNotFoundError:
    print(f"\nError: Could not find training data in '{ML_DIR}'.")
    exit()

print(f"Loaded original training data. Shape: {D_train.shape}")
n_samples, m_voxels, n_beamlets = D_train.shape

# --- HELPER FUNCTION: Calculate Objective ---
def calculate_objective(D, organ_sets, w):
    """
    Recalculates the objective value mirroring the solve_forward_problem logic:
    1. Quadratic: Square of the sum of violations
    2. Linear: Sum of violations
    3. Exponential: Scaled exp of max dose
    """
    alpha_1 = 0.001
    
    # Calculate Dose
    dose = D @ w
    
    # 1. OAR1 (Quadratic penalty - Square of the Sum)
    diff1 = np.maximum(0, dose[organ_sets['oar1']] - PARAMS['theta_thresh_1'])
    term1 = alpha_1 * (np.sum(diff1)**2)
    
    # 2. OAR2 (Linear penalty)
    diff2 = np.maximum(0, dose[organ_sets['oar2']] - PARAMS['theta_thresh_2'])
    term2 = np.sum(diff2)
    
    # 3. OAR3 (Scaled Exponential penalty)
    term3 = 0
    if len(organ_sets['oar3']) > 0:
        d_max_val = np.max(dose[organ_sets['oar3']])
        # Stable exponential calculation
        term3 = PARAMS['alpha_exp'] * np.exp(d_max_val / PARAMS['exp_scale_k'])
    
    return term1 + term2 + term3

# =============================================================================
# 3. Load Data Point 1: Min Penalty (Found "Best Case")
# =============================================================================
print(f"\nLoading 'Min Penalty' (Best Case) from: {BEST_CASE_DIR}")

try:
    D_min = np.load(f'{BEST_CASE_DIR}/D_best_case.npy')
    w_min = np.load(f'{BEST_CASE_DIR}/w_opt_best_case.npy')
    organ_sets_min_loaded = np.load(f'{BEST_CASE_DIR}/organ_sets_best_case.npy', allow_pickle=True)
    
    anatomy_min = organ_sets_min_loaded.item() if organ_sets_min_loaded.ndim == 0 else organ_sets_min_loaded

    obj_min = calculate_objective(D_min, anatomy_min, w_min)
    print(f"  > Best Case objective (calculated): {obj_min:.4f}")

except FileNotFoundError as e:
    print(f"\nCRITICAL ERROR: Could not find Best Case file: {e.filename}")
    exit()

# =============================================================================
# 4. Load Data Point 2: Max Penalty (Found "Worst Case")
# =============================================================================
print(f"\nLoading 'Max Penalty' (Worst Case) from: {WORST_CASE_DIR}")

try:
    D_max = np.load(f'{WORST_CASE_DIR}/D_worst_case.npy')
    w_max = np.load(f'{WORST_CASE_DIR}/w_opt_worst_case.npy')
    organ_sets_max_loaded = np.load(f'{WORST_CASE_DIR}/organ_sets_worst_case.npy', allow_pickle=True)
    
    anatomy_max = organ_sets_max_loaded.item() if organ_sets_max_loaded.ndim == 0 else organ_sets_max_loaded

    obj_max = calculate_objective(D_max, anatomy_max, w_max)
    print(f"  > Worst Case objective (calculated): {obj_max:.4f}")

except FileNotFoundError as e:
    print(f"\nCRITICAL ERROR: Could not find Worst Case file: {e.filename}")
    exit()

# =============================================================================
# 5. Reshape and Stack New Data
# =============================================================================
print("\nStacking and concatenating data...")

D_to_add = np.stack([D_min, D_max], axis=0) 
w_to_add = np.stack([w_min, w_max], axis=0)
organ_sets_to_add = np.array([anatomy_min, anatomy_max], dtype=object)
objectives_to_add = np.array([obj_min, obj_max])
ids_to_add = np.array([-1, -2]) 

# Concatenate to the BEGINNING
D_train_new = np.concatenate((D_to_add, D_train), axis=0)
w_train_new = np.concatenate((w_to_add, w_train), axis=0)
organ_sets_train_new = np.concatenate((organ_sets_to_add, organ_sets_train), axis=0)
objectives_train_new = np.concatenate((objectives_to_add, objectives_train), axis=0)
original_ids_train_new = np.concatenate((ids_to_add, original_ids_train), axis=0)

# =============================================================================
# 6. Save
# =============================================================================
np.save(f'{ML_DIR}/D_train.npy', D_train_new)
np.save(f'{ML_DIR}/w_train.npy', w_train_new)
np.save(f'{ML_DIR}/organ_sets_train.npy', organ_sets_train_new, allow_pickle=True)
np.save(f'{ML_DIR}/objectives_train.npy', objectives_train_new)
np.save(f'{ML_DIR}/original_ids_train.npy', original_ids_train_new)

print(f"Success. New training size: {len(D_train_new)}")
print("--- Augmentation Complete ---")

In [ ]:
#!/usr/bin/env python
# ============================================================================
# Smooth Additive Inverse Optimization for IMRT
#
# Uses vectorized constraint definitions for the Smoothness (C5)
# components to improve model generation speed for larger N.
# ============================================================================

import numpy as np
import cvxpy as cp
import time
import os

# ----------------------------------------------------------------------
# 0. CONFIGURATION & CRITICAL PARAMETERS
# ----------------------------------------------------------------------

PARAMS = {
    'theta_prescribed': 78.0,
    'theta_max_PTV': 81.9,
    'theta_thresh_1': 59.15,
    'theta_thresh_2': 59.15,
    'beta_1': 0.5,
    'beta_2': 1.5
}

NORMALIZATION_SETTINGS = {
    'idx_norm_zero': 0,
    'idx_norm_max': 1,
    'U_MAX': 1000.0 
}

# --- Model Configuration ---
ML_DIR = f"./ml_data_high_std" 
RESULTS_DIR = f"./results_high_std"
TRAINING_SIZES = [20, 40, 60, 80, 100, 120, 140, 160, 180, 200] 

BETA_0 = 2000.0       # Stage 1: "Large" beta
EPSILON_TOL = 1e-6    # Tolerance added to epsilon*
BISECT_GAP_TOL = 0.11    # Stopping tolerance for bisection
BISECT_BETA_LOW = 0.01   # Lower bound for bisection

# --- Solver Configuration ---
SOLVER = "MOSEK"
SOLVER_VERBOSE = False
CUSTOM_TOLERANCES = {
    'MSK_DPAR_INTPNT_CO_TOL_PFEAS': 1e-6,
    'MSK_DPAR_INTPNT_CO_TOL_DFEAS': 1e-6,
    'MSK_DPAR_INTPNT_CO_TOL_REL_GAP': 1e-6
}

# ----------------------------------------------------------------------
# 1. DATA LOADING & PRE-PROCESSING
# ----------------------------------------------------------------------

def load_data(data_dir, N):
    """Loads dataset and returns first N points."""
    print(f"--- Loading first {N} data points from {data_dir} ---")
    try:
        D_train_all = np.load(f'{data_dir}/D_train.npy', allow_pickle=True)
        w_train_all = np.load(f'{data_dir}/w_train.npy', allow_pickle=True)
        organ_sets_train_all = np.load(f'{data_dir}/organ_sets_train.npy', allow_pickle=True)
        
        D_train = D_train_all[:N]
        w_train = w_train_all[:N]
        organ_sets_train = organ_sets_train_all[:N]
        
        if len(D_train) < N:
            print(f"Warning: Requested {N}, loaded {len(D_train)}.")
            N = len(D_train)

        # Validation
        if N > 0:
            test_keys = organ_sets_train[0].keys()
            required = ['ptv', 'oar1', 'oar2', 'oar3']
            missing = [k for k in required if k not in test_keys]
            if missing:
                print(f"CRITICAL WARNING: Missing keys {missing} in organ_sets.")
        
        return D_train, w_train, organ_sets_train

    except Exception as e:
        print(f"Error loading data: {e}")
        raise

def preprocess_outcomes(w_obs, D_obs, organ_sets_obs, params):
    """Calculates observed outcomes Z_hat."""
    N = len(w_obs)
    K = 3
    Z_hat = np.zeros((N, K))
    
    print("--- Pre-processing observed outcomes (Z_hat) ---")
    for i in range(N):
        w_i = w_obs[i]
        D_i = D_obs[i]
        org_i = organ_sets_obs[i]
        d_dose = D_i @ w_i
        
        v_oar1 = org_i.get('oar1', np.array([]))
        v_oar2 = org_i.get('oar2', np.array([]))
        v_oar3 = org_i.get('oar3', np.array([]))
        
        if v_oar1.size > 0:
            Z_hat[i, 0] = np.sum(np.maximum(0, d_dose[v_oar1] - params['theta_thresh_1']))
        if v_oar2.size > 0:
            Z_hat[i, 1] = np.sum(np.maximum(0, d_dose[v_oar2] - params['theta_thresh_2']))
        if v_oar3.size > 0:
            Z_hat[i, 2] = np.max(d_dose[v_oar3])
            
    return Z_hat

# ----------------------------------------------------------------------
# 2. CVXPY MODEL HELPER FUNCTIONS
# ----------------------------------------------------------------------

def get_model_variables(N, K, D_obs, organ_sets_obs):
    """
    Initializes CVXPY variables.
    Note: Dual variables must be lists because voxel counts vary per patient.
    """
    delta = cp.Variable((N, K), nonneg=True, name="delta")
    delta_global = cp.Variable(N, name="delta_global") 
    lambdas = cp.Variable((N, K), name="lambda")
    epsilon = cp.Variable(nonneg=True, name="epsilon")
    
    # Dual variables (List of variables due to ragged dimensions)
    dual_vars = {
        "mu_1": [], "mu_2": [], "nu_1": [], "nu_2": [], 
        "pi": [], "sigma_1": [], "sigma_2": []
    }
    
    for i in range(N):
        n_b = D_obs[i].shape[1]
        org = organ_sets_obs[i]
        
        dual_vars["mu_1"].append(cp.Variable(len(org.get('ptv', [])), nonneg=True))
        dual_vars["mu_2"].append(cp.Variable(len(org.get('ptv', [])), nonneg=True))
        dual_vars["nu_1"].append(cp.Variable(len(org.get('oar1', [])), nonneg=True))
        dual_vars["nu_2"].append(cp.Variable(len(org.get('oar2', [])), nonneg=True))
        dual_vars["pi"].append(cp.Variable(len(org.get('oar3', [])), nonneg=True))
        dual_vars["sigma_1"].append(cp.Variable(n_b, nonneg=True))
        dual_vars["sigma_2"].append(cp.Variable(n_b, nonneg=True))

    return {
        "delta": delta, "delta_global": delta_global, 
        "lambdas": lambdas, "epsilon": epsilon, **dual_vars
    }

def build_constraints(variables, Z_hat, D_obs, organ_sets_obs, params, 
                      beta_val, epsilon_val, norm_settings):
    """
    Builds constraints. Uses vectorized operations for C5 (Smoothness).
    """
    delta = variables["delta"]
    delta_global = variables["delta_global"]
    lambdas = variables["lambdas"]
    
    N, K = Z_hat.shape
    cons = []
    
    if beta_val <= 0: raise ValueError("beta_val must be positive.")
    beta_inv = 1.0 / (2.0 * beta_val)
    
    # === C5: Smoothness & Convexity (VECTORIZED) ===
    # Using array slicing to avoid Python loops for N
    for k in range(K):
        # 1. Sort indices based on data Z_hat for outcome k
        idx_ord = np.argsort(Z_hat[:, k])
        
        # 2. Create index arrays for current (i) and next (j) neighbors
        i_indices = idx_ord[:-1] # Indices 0 to N-2
        j_indices = idx_ord[1:]  # Indices 1 to N-1
        
        # 3. Calculate constant data difference vector
        dz_vec = Z_hat[j_indices, k] - Z_hat[i_indices, k]
        
        # 4. Get variables aligned to sorted order
        delta_i = delta[i_indices, k]
        delta_j = delta[j_indices, k]
        lam_i   = lambdas[i_indices, k]
        lam_j   = lambdas[j_indices, k]
        
        # 5. Quadratic Term (Vectorized)
        lam_diff = lam_i - lam_j
        q_term = beta_inv * cp.square(lam_diff)
        
        # 6. Add Vectorized Constraints
        # (C5-2)
        cons.append(
            delta_j - delta_i - cp.multiply(lam_i, dz_vec) >= q_term
        )
        
        # (C5-1)
        cons.append(
            delta_i - delta_j - cp.multiply(lam_j, -dz_vec) >= q_term
        )
            
    # === C6: Monotonicity ===
    cons.append(lambdas >= 0)
    
    # === C7: Normalization ===
    cons.append(delta_global == cp.sum(delta, axis=1))
    
    idx_zero = norm_settings['idx_norm_zero']
    idx_max = norm_settings['idx_norm_max']
    
    if idx_zero < N: cons.append(delta_global[idx_zero] == 0)
    # Equality constraint (pinned point)
    if idx_max < N:  cons.append(delta_global[idx_max] == norm_settings['U_MAX'])

    # === C8 & Dual Constraints (Patient-specific Loop) ===
    for i in range(N):
        lam_i = lambdas[i, :]
        z_hat_i = Z_hat[i, :]
        D_i = D_obs[i]
        org_i = organ_sets_obs[i]
        n_b = D_i.shape[1]

        # Retrieve dual vars
        mu_1_i = variables["mu_1"][i]
        mu_2_i = variables["mu_2"][i]
        nu_1_i = variables["nu_1"][i]
        nu_2_i = variables["nu_2"][i]
        pi_i   = variables["pi"][i]
        sig_1_i = variables["sigma_1"][i]
        sig_2_i = variables["sigma_2"][i]
        
        v_ptv = org_i.get('ptv', np.array([]))
        v_oar1 = org_i.get('oar1', np.array([]))
        v_oar2 = org_i.get('oar2', np.array([]))
        v_oar3 = org_i.get('oar3', np.array([]))

        # --- Dual Objective Calculation ---
        obj_terms = []
        if v_ptv.size > 0: 
            obj_terms.append(cp.sum(mu_1_i * params['theta_prescribed'] - mu_2_i * params['theta_max_PTV']))
        if v_oar1.size > 0:
            obj_terms.append(-cp.sum(nu_1_i * params['theta_thresh_1']))
        if v_oar2.size > 0:
            obj_terms.append(-cp.sum(nu_2_i * params['theta_thresh_2']))
            
        dual_obj_i = sum(obj_terms)
        
        # C8
        cons.append(lam_i @ z_hat_i - dual_obj_i <= epsilon_val)
        
        # Dual Bounds (D1-D3)
        if v_oar1.size > 0: cons.append(nu_1_i <= lam_i[0])
        if v_oar2.size > 0: cons.append(nu_2_i <= lam_i[1])
        if v_oar3.size > 0: cons.append(cp.sum(pi_i) <= lam_i[2])
            
        # Stationarity (D4)
        grad_sum = -sig_1_i + sig_2_i
        grad_sum += (params['beta_1'] / n_b) * cp.sum(sig_1_i)
        grad_sum -= (params['beta_2'] / n_b) * cp.sum(sig_2_i)
        
        if v_ptv.size > 0:  grad_sum -= (mu_1_i - mu_2_i) @ D_i[v_ptv, :]
        if v_oar1.size > 0: grad_sum += nu_1_i @ D_i[v_oar1, :]
        if v_oar2.size > 0: grad_sum += nu_2_i @ D_i[v_oar2, :]
        if v_oar3.size > 0: grad_sum += pi_i @ D_i[v_oar3, :]
            
        cons.append(grad_sum >= 0)
        
    return cons

# ----------------------------------------------------------------------
# 3. BISECTION SEARCH HELPER
# ----------------------------------------------------------------------

def is_feasible(beta_val, Z_hat, D_obs, organ_sets_obs, params, epsilon_feas, norm_settings, max_retries=3):
    if beta_val <= 0: return False
    
    N, K = Z_hat.shape
    current_beta = beta_val
    
    for retry in range(max_retries):
        try:
            vars_feas = get_model_variables(N, K, D_obs, organ_sets_obs)
            cons_feas = build_constraints(
                vars_feas, Z_hat, D_obs, organ_sets_obs, 
                params, current_beta, epsilon_feas, norm_settings
            )
            prob = cp.Problem(cp.Minimize(0), cons_feas)
            prob.solve(solver=SOLVER, verbose=SOLVER_VERBOSE, mosek_params=CUSTOM_TOLERANCES)
            
            if prob.status in {"optimal", "optimal_inaccurate"}:
                return True
            
            current_beta *= 1.1 # Retry with relaxed beta
            
        except cp.SolverError:
            current_beta *= 1.1
            
    return False

def find_beta_bisect(Z_hat, D_obs, organ_sets_obs, params, epsilon_val,
                     beta_init, beta_low, gap_tol, norm_settings):
    """Performs bisection search to find the minimum feasible beta."""
    print("\n--- STAGE 2: Finding minimum beta* (Bisection Search) ---")
    
    beta_high = beta_init
    # 1. Establish Feasible Upper Bound
    while not is_feasible(beta_high, Z_hat, D_obs, organ_sets_obs, params, epsilon_val, norm_settings):
        print(f"  Beta = {beta_high:.4f} INFEASIBLE. Doubling...")
        beta_high *= 2.0
        if beta_high > 1e9: raise RuntimeError("Model appears infeasible even at large beta.")
        
    print(f"  Feasible upper bound: {beta_high:.4f}")
    
    # 2. Bisection
    while beta_high - beta_low > gap_tol:
        beta_mid = 0.5 * (beta_low + beta_high)
        if is_feasible(beta_mid, Z_hat, D_obs, organ_sets_obs, params, epsilon_val, norm_settings):
            beta_high = beta_mid # Feasible, try lower
        else:
            beta_low = beta_mid  # Infeasible, need higher
            
        print(f"  Range: [{beta_low:.2f}, {beta_high:.2f}]")
        
    return beta_high

# ----------------------------------------------------------------------
# 4. OPTIMIZATION RUNNER
# ----------------------------------------------------------------------

def run_optimization_for_N(N_obs):
    """Runs the 3-stage optimization for N_obs."""
    start_time = time.time()
    
    # 1. Load Data
    try:
        D_obs, w_obs, organ_sets_obs = load_data(ML_DIR, N_obs)
        N_obs = len(D_obs) # Adjust if fewer loaded
    except Exception:
        return

    if N_obs <= 1:
        print("Insufficient data for normalization. Skipping.")
        return

    # 2. Preprocess
    Z_hat_obs = preprocess_outcomes(w_obs, D_obs, organ_sets_obs, PARAMS)
    K_outcomes = Z_hat_obs.shape[1]

    # --- STAGE 1: Minimize Epsilon ---
    print("\n--- STAGE 1: Minimizing Epsilon ---")
    vars_s1 = get_model_variables(N_obs, K_outcomes, D_obs, organ_sets_obs)
    epsilon_s1_var = vars_s1["epsilon"]
    
    cons_s1 = build_constraints(
        vars_s1, Z_hat_obs, D_obs, organ_sets_obs, PARAMS, 
        beta_val=BETA_0, epsilon_val=epsilon_s1_var, norm_settings=NORMALIZATION_SETTINGS
    )
    
    prob_s1 = cp.Problem(cp.Minimize(epsilon_s1_var), cons_s1)
    prob_s1.solve(solver=SOLVER, verbose=SOLVER_VERBOSE, mosek_params=CUSTOM_TOLERANCES)
    
    if prob_s1.status != "optimal":
        print(f"Stage 1 failed: {prob_s1.status}")
        return

    epsilon_star = epsilon_s1_var.value
    epsilon_s2 = epsilon_star + EPSILON_TOL
    print(f"  epsilon* = {epsilon_star:.8f}")

    # --- STAGE 2: Find Beta* ---
    beta_star = find_beta_bisect(
        Z_hat_obs, D_obs, organ_sets_obs, PARAMS,
        epsilon_val=epsilon_s2,
        beta_init=10, 
        beta_low=BISECT_BETA_LOW,
        gap_tol=BISECT_GAP_TOL,
        norm_settings=NORMALIZATION_SETTINGS
    )
    
    # --- STAGE 3: Maximize Deltas (WITH RETRY) ---
    print(f"\n--- STAGE 3: Maximizing Deltas (Initial beta={beta_star:.2f}) ---")
    
    current_beta_s3 = beta_star
    max_stage3_attempts = 10
    stage3_success = False
    
    # Loop for retry logic
    for attempt in range(max_stage3_attempts):
        print(f"  > Attempt {attempt+1}/{max_stage3_attempts} with beta={current_beta_s3:.4f}...")
        
        try:
            # Re-initialize variables and constraints with current beta
            vars_s3 = get_model_variables(N_obs, K_outcomes, D_obs, organ_sets_obs)
            
            cons_s3 = build_constraints(
                vars_s3, Z_hat_obs, D_obs, organ_sets_obs, PARAMS, 
                beta_val=current_beta_s3, epsilon_val=epsilon_s2, norm_settings=NORMALIZATION_SETTINGS
            )
            
            prob_s3 = cp.Problem(cp.Maximize(cp.sum(vars_s3["delta_global"])), cons_s3)
            prob_s3.solve(solver=SOLVER, verbose=SOLVER_VERBOSE, mosek_params=CUSTOM_TOLERANCES)
            
            if prob_s3.status in ["optimal", "optimal_inaccurate"]:
                print(f"    -> Success! Status: {prob_s3.status}")
                stage3_success = True
                break
            else:
                print(f"    -> Failed with status: {prob_s3.status}")
                # Raise error to trigger the except block logic below
                raise cp.SolverError(f"Solver returned {prob_s3.status}")

        except (cp.SolverError, Exception) as e:
            print(f"    -> Error occurred: {str(e)[:100]}")
            print("    -> Increasing beta by 50% and retrying.")
            current_beta_s3 *= 1.5
    
    if not stage3_success:
        print("CRITICAL ERROR: Stage 3 failed after all retry attempts.")
        return

    # --- Save Results ---
    os.makedirs(RESULTS_DIR, exist_ok=True)
    np.save(f'{RESULTS_DIR}/final_delta_global_N{N_obs}_add_smooth.npy', vars_s3["delta_global"].value)
    np.save(f'{RESULTS_DIR}/final_delta_comp_N{N_obs}_add_smooth.npy', vars_s3["delta"].value)
    np.save(f'{RESULTS_DIR}/final_lambda_N{N_obs}_add_smooth.npy', vars_s3["lambdas"].value)
    np.save(f'{RESULTS_DIR}/Z_hat_obs_N{N_obs}_add_smooth.npy', Z_hat_obs)
    
    # Save the beta used in Stage 3
    np.save(f'{RESULTS_DIR}/beta_star_N{N_obs}_add_smooth.npy', np.array([current_beta_s3]))
    np.save(f'{RESULTS_DIR}/epsilon_star_N{N_obs}_add_smooth.npy', np.array([epsilon_star]))

    final_delta_global = vars_s3["delta_global"].value
    final_delta_comp = vars_s3["delta"].value
    final_lambda = vars_s3["lambdas"].value

    print("\nFinal Global Delta (f(z_i)):")
    print(np.round(final_delta_global, 4))
    
    max_global_val = np.max(final_delta_global)
    print(f"\n[DIAGNOSTIC] Max Delta Global: {max_global_val:.4f}") 
    print(f"             (Target limit was: {NORMALIZATION_SETTINGS['U_MAX']})") 

    print("\nFinal Component Delta (f_k(z_i,k)):")
    print(np.round(final_delta_comp, 4))

    print("\nFinal Lambda (f'_k(z_i,k)):")
    print(np.round(final_lambda, 4))

    print(f"Done in {time.time() - start_time:.2f}s. Results saved.")

def main():
    os.makedirs(RESULTS_DIR, exist_ok=True)
    print("=== Starting Smooth Additive Inverse Optimization ===")
    for n in TRAINING_SIZES:
        run_optimization_for_N(n)
    print("\nAll runs complete.")

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python
# ============================================================================
# Pure LP Additive Inverse Optimization for IMRT (Gurobi Version)
#
# Pure LP: no smoothness (1/beta) terms.
# Enforces non-smooth convexity constraints directly.
# Solves using Gurobi.
# ============================================================================

import numpy as np
import cvxpy as cp
import time
import os

# ----------------------------------------------------------------------
# 0. CONFIGURATION & CRITICAL PARAMETERS
# ----------------------------------------------------------------------

PARAMS = {
    'theta_prescribed': 78.0,
    'theta_max_PTV': 81.9,
    'theta_thresh_1': 59.15,
    'theta_thresh_2': 59.15,
    'beta_1': 0.5,
    'beta_2': 1.5
}

NORMALIZATION_SETTINGS = {
    'idx_norm_zero': 0,
    'idx_norm_max': 1,
    'U_MAX': 1000.0 
}

# --- Model Configuration ---
ML_DIR = f"./ml_data_high_std" 
RESULTS_DIR = f"./results_high_std_add_nonsmooth_lp"
TRAINING_SIZES = [20, 40, 60, 80, 100, 120, 140, 160, 180, 200] 

EPSILON_TOL = 1e-6    # Tolerance added to epsilon*

# --- Solver Configuration ---
SOLVER = "GUROBI"
SOLVER_VERBOSE = False
# Custom tolerances can be passed to Gurobi if needed via solver_opts
# e.g., 'FeasibilityTol': 1e-6, 'OptimalityTol': 1e-6

# ----------------------------------------------------------------------
# 1. DATA LOADING & PRE-PROCESSING
# ----------------------------------------------------------------------

def load_data(data_dir, N):
    """Loads dataset and returns first N points."""
    print(f"--- Loading first {N} data points from {data_dir} ---")
    try:
        D_train_all = np.load(f'{data_dir}/D_train.npy', allow_pickle=True)
        w_train_all = np.load(f'{data_dir}/w_train.npy', allow_pickle=True)
        organ_sets_train_all = np.load(f'{data_dir}/organ_sets_train.npy', allow_pickle=True)
        
        D_train = D_train_all[:N]
        w_train = w_train_all[:N]
        organ_sets_train = organ_sets_train_all[:N]
        
        if len(D_train) < N:
            print(f"Warning: Requested {N}, loaded {len(D_train)}.")
            N = len(D_train)

        if N > 0:
            test_keys = organ_sets_train[0].keys()
            required = ['ptv', 'oar1', 'oar2', 'oar3']
            missing = [k for k in required if k not in test_keys]
            if missing:
                print(f"CRITICAL WARNING: Missing keys {missing} in organ_sets.")
        
        return D_train, w_train, organ_sets_train

    except Exception as e:
        print(f"Error loading data: {e}")
        raise

def preprocess_outcomes(w_obs, D_obs, organ_sets_obs, params):
    """Calculates observed outcomes Z_hat."""
    N = len(w_obs)
    K = 3
    Z_hat = np.zeros((N, K))
    
    print("--- Pre-processing observed outcomes (Z_hat) ---")
    for i in range(N):
        w_i = w_obs[i]
        D_i = D_obs[i]
        org_i = organ_sets_obs[i]
        d_dose = D_i @ w_i
        
        v_oar1 = org_i.get('oar1', np.array([]))
        v_oar2 = org_i.get('oar2', np.array([]))
        v_oar3 = org_i.get('oar3', np.array([]))
        
        if v_oar1.size > 0:
            Z_hat[i, 0] = np.sum(np.maximum(0, d_dose[v_oar1] - params['theta_thresh_1']))
        if v_oar2.size > 0:
            Z_hat[i, 1] = np.sum(np.maximum(0, d_dose[v_oar2] - params['theta_thresh_2']))
        if v_oar3.size > 0:
            Z_hat[i, 2] = np.max(d_dose[v_oar3])
            
    return Z_hat

# ----------------------------------------------------------------------
# 2. CVXPY MODEL HELPER FUNCTIONS (PURE LP)
# ----------------------------------------------------------------------

def get_model_variables(N, K, D_obs, organ_sets_obs):
    """
    Initializes CVXPY variables.
    """
    delta = cp.Variable((N, K), nonneg=True, name="delta")
    delta_global = cp.Variable(N, name="delta_global") 
    lambdas = cp.Variable((N, K), name="lambda")
    epsilon = cp.Variable(nonneg=True, name="epsilon")
    
    # Dual variables (List of variables due to ragged dimensions)
    dual_vars = {
        "mu_1": [], "mu_2": [], "nu_1": [], "nu_2": [], 
        "pi": [], "sigma_1": [], "sigma_2": []
    }
    
    for i in range(N):
        n_b = D_obs[i].shape[1]
        org = organ_sets_obs[i]
        
        dual_vars["mu_1"].append(cp.Variable(len(org.get('ptv', [])), nonneg=True))
        dual_vars["mu_2"].append(cp.Variable(len(org.get('ptv', [])), nonneg=True))
        dual_vars["nu_1"].append(cp.Variable(len(org.get('oar1', [])), nonneg=True))
        dual_vars["nu_2"].append(cp.Variable(len(org.get('oar2', [])), nonneg=True))
        dual_vars["pi"].append(cp.Variable(len(org.get('oar3', [])), nonneg=True))
        dual_vars["sigma_1"].append(cp.Variable(n_b, nonneg=True))
        dual_vars["sigma_2"].append(cp.Variable(n_b, nonneg=True))

    return {
        "delta": delta, "delta_global": delta_global, 
        "lambdas": lambdas, "epsilon": epsilon, **dual_vars
    }

def build_constraints(variables, Z_hat, D_obs, organ_sets_obs, params, 
                      epsilon_val, norm_settings):
    """
    Builds constraints for Pure LP.
    No 1/beta quadratic terms in C5.
    """
    delta = variables["delta"]
    delta_global = variables["delta_global"]
    lambdas = variables["lambdas"]
    
    N, K = Z_hat.shape
    cons = []
    
    # === C5: Non-Smooth Convexity (Pure LP) ===
    # Vectorized implementation
    for k in range(K):
        # 1. Sort indices based on data Z_hat for outcome k
        idx_ord = np.argsort(Z_hat[:, k])
        
        # 2. Create index arrays for current (i) and next (j) neighbors
        i_indices = idx_ord[:-1] # Indices 0 to N-2
        j_indices = idx_ord[1:]  # Indices 1 to N-1
        
        # 3. Calculate constant data difference vector
        dz_vec = Z_hat[j_indices, k] - Z_hat[i_indices, k]
        
        # 4. Get variables aligned to sorted order
        delta_i = delta[i_indices, k]
        delta_j = delta[j_indices, k]
        lam_i   = lambdas[i_indices, k]
        lam_j   = lambdas[j_indices, k]
        
        # 5. Add Linear Constraints (No q_term)
        # f(z_j) >= f(z_i) + subgrad * (z_j - z_i)
        # (C5-2)
        cons.append(
            delta_j - delta_i - cp.multiply(lam_i, dz_vec) >= 0
        )
        
        # (C5-1)
        cons.append(
            delta_i - delta_j - cp.multiply(lam_j, -dz_vec) >= 0
        )
            
    # === C6: Monotonicity ===
    cons.append(lambdas >= 0)
    
    # === C7: Normalization ===
    cons.append(delta_global == cp.sum(delta, axis=1))
    
    idx_zero = norm_settings['idx_norm_zero']
    idx_max = norm_settings['idx_norm_max']
    
    if idx_zero < N: cons.append(delta_global[idx_zero] == 0)
    if idx_max < N:  cons.append(delta_global[idx_max] == norm_settings['U_MAX'])
    
    # === C8 & Dual Constraints (Patient-specific Loop) ===
    for i in range(N):
        lam_i = lambdas[i, :]
        z_hat_i = Z_hat[i, :]
        D_i = D_obs[i]
        org_i = organ_sets_obs[i]
        n_b = D_i.shape[1]

        # Retrieve dual vars
        mu_1_i = variables["mu_1"][i]
        mu_2_i = variables["mu_2"][i]
        nu_1_i = variables["nu_1"][i]
        nu_2_i = variables["nu_2"][i]
        pi_i   = variables["pi"][i]
        sig_1_i = variables["sigma_1"][i]
        sig_2_i = variables["sigma_2"][i]
        
        v_ptv = org_i.get('ptv', np.array([]))
        v_oar1 = org_i.get('oar1', np.array([]))
        v_oar2 = org_i.get('oar2', np.array([]))
        v_oar3 = org_i.get('oar3', np.array([]))

        # --- Dual Objective Calculation ---
        obj_terms = []
        if v_ptv.size > 0: 
            obj_terms.append(cp.sum(mu_1_i * params['theta_prescribed'] - mu_2_i * params['theta_max_PTV']))
        if v_oar1.size > 0:
            obj_terms.append(-cp.sum(nu_1_i * params['theta_thresh_1']))
        if v_oar2.size > 0:
            obj_terms.append(-cp.sum(nu_2_i * params['theta_thresh_2']))
            
        dual_obj_i = sum(obj_terms)
        
        # C8: Primal-Dual Gap <= Epsilon
        cons.append(lam_i @ z_hat_i - dual_obj_i <= epsilon_val)
        
        # Dual Bounds (D1-D3)
        if v_oar1.size > 0: cons.append(nu_1_i <= lam_i[0])
        if v_oar2.size > 0: cons.append(nu_2_i <= lam_i[1])
        if v_oar3.size > 0: cons.append(cp.sum(pi_i) <= lam_i[2])
            
        # Stationarity (D4)
        grad_sum = -sig_1_i + sig_2_i
        
        # Add regularization terms
        grad_sum += (params['beta_1'] / n_b) * cp.sum(sig_1_i)
        grad_sum -= (params['beta_2'] / n_b) * cp.sum(sig_2_i)
        
        if v_ptv.size > 0:  grad_sum -= (mu_1_i - mu_2_i) @ D_i[v_ptv, :]
        if v_oar1.size > 0: grad_sum += nu_1_i @ D_i[v_oar1, :]
        if v_oar2.size > 0: grad_sum += nu_2_i @ D_i[v_oar2, :]
        if v_oar3.size > 0: grad_sum += pi_i @ D_i[v_oar3, :]
            
        cons.append(grad_sum >= 0)
        
    return cons

# ----------------------------------------------------------------------
# 4. OPTIMIZATION RUNNER
# ----------------------------------------------------------------------

def run_optimization_for_N(N_obs):
    """Runs the 2-stage optimization (Pure LP) for N_obs."""
    start_time = time.time()
    
    # 1. Load Data
    try:
        D_obs, w_obs, organ_sets_obs = load_data(ML_DIR, N_obs)
        N_obs = len(D_obs) # Adjust if fewer loaded
    except Exception:
        return

    if N_obs <= 1:
        print("Insufficient data for normalization. Skipping.")
        return

    # 2. Preprocess
    Z_hat_obs = preprocess_outcomes(w_obs, D_obs, organ_sets_obs, PARAMS)
    K_outcomes = Z_hat_obs.shape[1]

    # --- STAGE 1: Minimize Epsilon (Find Feasibility) ---
    print("\n--- STAGE 1: Minimizing Epsilon (Feasibility) ---")
    vars_s1 = get_model_variables(N_obs, K_outcomes, D_obs, organ_sets_obs)
    epsilon_s1_var = vars_s1["epsilon"]
    
    cons_s1 = build_constraints(
        vars_s1, Z_hat_obs, D_obs, organ_sets_obs, PARAMS, 
        epsilon_val=epsilon_s1_var, norm_settings=NORMALIZATION_SETTINGS
    )
    
    prob_s1 = cp.Problem(cp.Minimize(epsilon_s1_var), cons_s1)
    
    # Solve with Gurobi
    # You can pass Gurobi parameters like 'OutputFlag': 0 to silence output
    prob_s1.solve(solver=SOLVER, verbose=SOLVER_VERBOSE)
    
    if prob_s1.status != "optimal":
        print(f"Stage 1 failed: {prob_s1.status}")
        return

    epsilon_star = epsilon_s1_var.value
    # Add tolerance for Stage 2
    epsilon_s2 = epsilon_star + EPSILON_TOL
    print(f"  epsilon* = {epsilon_star:.8f}")

    # --- STAGE 2: Maximize Deltas (Pure LP) ---
    # In pure LP, we don't need to bisect Beta. We just maximize the objective
    # given the fixed epsilon constraint.
    print(f"\n--- STAGE 2: Maximizing Deltas (Pure LP) ---")
    vars_s2 = get_model_variables(N_obs, K_outcomes, D_obs, organ_sets_obs)
    
    cons_s2 = build_constraints(
        vars_s2, Z_hat_obs, D_obs, organ_sets_obs, PARAMS, 
        epsilon_val=epsilon_s2, norm_settings=NORMALIZATION_SETTINGS
    )
    
    prob_s2 = cp.Problem(cp.Maximize(cp.sum(vars_s2["delta_global"])), cons_s2)
    prob_s2.solve(solver=SOLVER, verbose=SOLVER_VERBOSE)
    
    if prob_s2.status != "optimal":
        print(f"Stage 2 failed: {prob_s2.status}")
        return

    # --- Save Results ---
    os.makedirs(RESULTS_DIR, exist_ok=True)
    np.save(f'{RESULTS_DIR}/final_delta_global_N{N_obs}_lp.npy', vars_s2["delta_global"].value)
    np.save(f'{RESULTS_DIR}/final_delta_comp_N{N_obs}_lp.npy', vars_s2["delta"].value)
    np.save(f'{RESULTS_DIR}/final_lambda_N{N_obs}_lp.npy', vars_s2["lambdas"].value)
    np.save(f'{RESULTS_DIR}/Z_hat_obs_N{N_obs}_lp.npy', Z_hat_obs)
    np.save(f'{RESULTS_DIR}/epsilon_star_N{N_obs}_lp.npy', np.array([epsilon_star]))

    # --- Results ---
    final_delta_global = vars_s2["delta_global"].value
    final_delta_comp = vars_s2["delta"].value
    final_lambda = vars_s2["lambdas"].value

    print("\nFinal Global Delta (f(z_i)):")
    print(np.round(final_delta_global, 4))
    
    max_global_val = np.max(final_delta_global)
    print(f"\n[DIAGNOSTIC] Max Delta Global: {max_global_val:.4f}") 
    print(f"             (Target limit was: {NORMALIZATION_SETTINGS['U_MAX']})") 

    print("\nFinal Component Delta (f_k(z_i,k)):")
    print(np.round(final_delta_comp, 4))

    print("\nFinal Lambda (f'_k(z_i,k)):")
    print(np.round(final_lambda, 4))

    print(f"Done in {time.time() - start_time:.2f}s. Results saved.")

def main():
    os.makedirs(RESULTS_DIR, exist_ok=True)
    print("=== Starting Pure LP Additive Inverse Optimization ===")
    for n in TRAINING_SIZES:
        run_optimization_for_N(n)
    print("\nAll runs complete.")

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python
# ============================================================================
# Non-Additive Smooth Model - INVERSE OPTIMIZATION TRAINING
# ============================================================================
# This script implements the THREE-STAGE NON-ADDITIVE SMOOTH IO model.
#
# STAGE 1: Minimize epsilon (suboptimality) using a large starting beta.
# STAGE 2: Use Bisection Search to find the minimum feasible smoothing parameter (beta*).
# STAGE 3: Maximize global delta (utility) using the final beta* and epsilon*.
# ============================================================================

import numpy as np
import cvxpy as cp
import time
import os
import traceback

# ----------------------------------------------------------------------
# 0. CONFIGURATION & CRITICAL PARAMETERS
# ----------------------------------------------------------------------

# Configuration
ML_DIR_NONADD_SMOOTH = "./ml_data_high_std"
RESULTS_DIR_NONADD_SMOOTH = "./results_high_std_non_add_smooth"
os.makedirs(RESULTS_DIR_NONADD_SMOOTH, exist_ok=True)

TRAINING_SIZES_NONADD_SMOOTH = [20, 40, 60, 80, 100, 120, 140, 160, 180, 200] 

# Smoothing parameters
BETA_0_NONADD_SMOOTH = 2000.0   # Stage 1: Large starting beta
EPSILON_TOL_NONADD_SMOOTH = 1e-6 # Tolerance added to epsilon* for Stages 2/3
BISECT_GAP_TOL = 0.2           # Bisection search stopping tolerance
BISECT_BETA_LOW = 0.01            # Assumed infeasible beta to start bisection

NORMALIZATION_SETTINGS_NONADD_SMOOTH = {
    'idx_norm_zero': 0,
    'idx_norm_max': 1,
    'U_MAX': 10000.0
}

PARAMS_NONADD_SMOOTH = {
    'theta_prescribed': 78.0,
    'theta_max_PTV': 81.9,
    'theta_thresh_1': 59.15,
    'theta_thresh_2': 59.15,
    'beta_1': 0.5,
    'beta_2': 1.5
}

SOLVER_NONADD_SMOOTH = "MOSEK"
SOLVER_VERBOSE_NONADD_SMOOTH = False

# ----------------------------------------------------------------------
# 1. HELPER FUNCTIONS FOR DATA AND MODEL
# ----------------------------------------------------------------------

def load_data_nonadd_smooth(data_dir, n_points):
    """Load training data."""
    try:
        D_all = np.load(f'{data_dir}/D_train.npy', allow_pickle=True)
        w_all = np.load(f'{data_dir}/w_train.npy', allow_pickle=True)
        organ_sets_all = np.load(f'{data_dir}/organ_sets_train.npy', allow_pickle=True)
        n_use = min(n_points, len(D_all))
        return D_all[:n_use], w_all[:n_use], organ_sets_all[:n_use]
    except Exception as e:
        print(f"Error loading data: {e}")
        raise

def preprocess_outcomes_nonadd_smooth(w_obs, D_obs, organ_sets_obs, params):
    """Calculates Z_hat (OAR overdose sums and max dose) for non-additive model."""
    N = len(w_obs)
    K = 3
    Z_hat = np.zeros((N, K))
    for i in range(N):
        d_dose = D_obs[i] @ w_obs[i]
        org_i = organ_sets_obs[i]
        v_oar1 = org_i.get('oar1', np.array([]))
        v_oar2 = org_i.get('oar2', np.array([]))
        v_oar3 = org_i.get('oar3', np.array([]))
        
        if v_oar1.size > 0:
            overdose_1 = np.maximum(0, d_dose[v_oar1] - params['theta_thresh_1'])
            Z_hat[i, 0] = np.sum(overdose_1)
        if v_oar2.size > 0:
            overdose_2 = np.maximum(0, d_dose[v_oar2] - params['theta_thresh_2'])
            Z_hat[i, 1] = np.sum(overdose_2)
        if v_oar3.size > 0:
            Z_hat[i, 2] = np.max(d_dose[v_oar3])
    return Z_hat

def get_model_variables_nonadd(N, K, D_obs, organ_sets_obs):
    """Create CVXPY variables for NON-ADDITIVE model."""
    delta_global = cp.Variable(N, nonneg=True, name="delta_global")
    lambdas = cp.Variable((N, K), name="lambda")
    epsilon = cp.Variable(nonneg=True, name="epsilon")
    
    # Dual variables (must be lists due to ragged D dimensions)
    mu_1, mu_2, nu_1, nu_2, pi, sigma_1, sigma_2 = [], [], [], [], [], [], []
    for i in range(N):
        D_i = D_obs[i]
        org_i = organ_sets_obs[i]
        n_b = D_i.shape[1]
        
        mu_1.append(cp.Variable(len(org_i.get('ptv', [])), nonneg=True))
        mu_2.append(cp.Variable(len(org_i.get('ptv', [])), nonneg=True))
        nu_1.append(cp.Variable(len(org_i.get('oar1', [])), nonneg=True))
        nu_2.append(cp.Variable(len(org_i.get('oar2', [])), nonneg=True))
        pi.append(cp.Variable(len(org_i.get('oar3', [])), nonneg=True))
        sigma_1.append(cp.Variable(n_b, nonneg=True))
        sigma_2.append(cp.Variable(n_b, nonneg=True))
    
    return {
        "delta_global": delta_global, "lambdas": lambdas, "epsilon": epsilon,
        "mu_1": mu_1, "mu_2": mu_2, "nu_1": nu_1, "nu_2": nu_2,
        "pi": pi, "sigma_1": sigma_1, "sigma_2": sigma_2
    }

def build_constraints_nonadd_smooth(variables, Z_hat, D_obs, organ_sets_obs, params,
                                     beta_val, epsilon_val, norm_settings):
    """
    Build NON-ADDITIVE SMOOTH constraints.
    Vectorized logic used for C5 to improve performance.
    """
    delta_global = variables["delta_global"]
    lambdas = variables["lambdas"]
    N, K = Z_hat.shape
    cons = []
    
    print(f"--- Building NON-ADDITIVE SMOOTH constraints (beta={beta_val}) ---")
    
    if beta_val <= 0:
        raise ValueError("beta_val must be positive for the smooth model.")
        
    beta_inv = 1.0 / (2.0 * beta_val)

    # === C5: NON-ADDITIVE SMOOTH Convexity (Vectorized) ===
    # We need to compare every i with every j (where i != j).
    # Index vectors over all pairs.
    
    # Create indices for all pairs (N*N pairs)
    idx_i_full = np.repeat(np.arange(N), N) # [0,0,0, 1,1,1, 2,2,2...]
    idx_j_full = np.tile(np.arange(N), N)   # [0,1,2, 0,1,2, 0,1,2...]
    
    # Filter out i == j to strictly handle distinct pairs
    mask = idx_i_full != idx_j_full
    idx_i = idx_i_full[mask]
    idx_j = idx_j_full[mask]
    
    # 1. Linear Term: lambda_i^T * (Z_j - Z_i)
    Z_diff_mat = Z_hat[idx_j] - Z_hat[idx_i] # Shape (N*(N-1), K)
    # Perform row-wise dot product efficiently
    linear_term = cp.sum(cp.multiply(lambdas[idx_i], Z_diff_mat), axis=1)
    
    # 2. Quadratic Term: (1/2b) * ||lambda_j - lambda_i||^2
    lambda_diff = lambdas[idx_j] - lambdas[idx_i] # Shape (N*(N-1), K)
    
    # Row-wise sum of squares
    q_term = beta_inv * cp.sum(cp.square(lambda_diff), axis=1)
    
    # 3. Add Vectorized Constraint
    # delta[j] - delta[i] - linear >= quadratic
    cons.append(
        delta_global[idx_j] - delta_global[idx_i] - linear_term >= q_term
    )
            
    # === C6: Monotonicity ===
    cons.append(lambdas >= 0)
    
    # === C7: Normalization ===
    idx_zero = norm_settings['idx_norm_zero']
    idx_max = norm_settings['idx_norm_max']
    u_val = norm_settings['U_MAX']
    
    if idx_zero < N: cons.append(delta_global[idx_zero] == 0)
    if idx_max < N:  cons.append(delta_global[idx_max] == u_val)
    # === C8: First-Order Sub-optimality ===
    # Looped per patient because D_i dimensions vary (ragged data)
    for i in range(N):
        lam_i = lambdas[i, :]
        z_hat_i = Z_hat[i, :]
        D_i = D_obs[i]
        n_b = D_i.shape[1]
        
        # Dual variables
        mu_1_i = variables["mu_1"][i]
        mu_2_i = variables["mu_2"][i]
        nu_1_i = variables["nu_1"][i]
        nu_2_i = variables["nu_2"][i]
        pi_i   = variables["pi"][i]
        sigma_1_i = variables["sigma_1"][i]
        sigma_2_i = variables["sigma_2"][i]
        
        # Calculate h(lambda) - dual objective
        h_i = 0
        if mu_1_i.size > 0: h_i += cp.sum(mu_1_i) * params['theta_prescribed']
        if mu_2_i.size > 0: h_i -= cp.sum(mu_2_i) * params['theta_max_PTV']
        if nu_1_i.size > 0: h_i -= cp.sum(nu_1_i) * params['theta_thresh_1']
        if nu_2_i.size > 0: h_i -= cp.sum(nu_2_i) * params['theta_thresh_2']
        
        # Constraint C8
        cons.append(lam_i @ z_hat_i - h_i <= epsilon_val)
        
        # Dual feasibility constraints
        if nu_1_i.size > 0: cons.append(nu_1_i <= lam_i[0])
        if nu_2_i.size > 0: cons.append(nu_2_i <= lam_i[1])
        if pi_i.size > 0:   cons.append(cp.sum(pi_i) <= lam_i[2])
            
        # Stationarity (D4)
        term_D_ptv = (mu_1_i - mu_2_i) @ D_i[organ_sets_obs[i].get('ptv',[]), :] if mu_1_i.size > 0 else 0
        term_D_oar1 = nu_1_i @ D_i[organ_sets_obs[i].get('oar1',[]), :] if nu_1_i.size > 0 else 0
        term_D_oar2 = nu_2_i @ D_i[organ_sets_obs[i].get('oar2',[]), :] if nu_2_i.size > 0 else 0
        term_D_oar3 = pi_i @ D_i[organ_sets_obs[i].get('oar3',[]), :] if pi_i.size > 0 else 0
            
        term_sigma_sum_1 = (params['beta_1'] / n_b) * cp.sum(sigma_1_i)
        term_sigma_sum_2 = (params['beta_2'] / n_b) * cp.sum(sigma_2_i)
        
        LHS_D4 = (-term_D_ptv + term_D_oar1 + term_D_oar2 + term_D_oar3
                  - sigma_1_i + term_sigma_sum_1
                  + sigma_2_i - term_sigma_sum_2)
        
        cons.append(LHS_D4 >= 0)
        
    print(f"    Built constraints (Vectorized C5 + {N} suboptimality blocks).")
    return cons

# ----------------------------------------------------------------------
# 2. BISECTION SEARCH HELPER FUNCTIONS
# ----------------------------------------------------------------------

def is_feasible(beta_val, Z_hat, D_obs, organ_sets_obs, params, epsilon_feas, norm_settings):
    """Checks if the model is feasible for a given beta and epsilon."""
    if beta_val <= 0: return False
    N, K = Z_hat.shape
    try:
        vars_feas = get_model_variables_nonadd(N, K, D_obs, organ_sets_obs)
        cons_feas = build_constraints_nonadd_smooth(
            vars_feas, Z_hat, D_obs, organ_sets_obs, 
            params, beta_val, epsilon_feas, norm_settings
        )
        prob = cp.Problem(cp.Minimize(0), cons_feas)
        prob.solve(solver=SOLVER_NONADD_SMOOTH, verbose=False)
        return prob.status in {"optimal", "optimal_inaccurate"}
    except Exception:
        return False

def find_beta_bisect(Z_hat, D_obs, organ_sets_obs, params, epsilon_val,
                     beta_init, beta_low, gap_tol, norm_settings):
    """Performs bisection search to find the minimum feasible beta*."""
    print("\n--- STAGE 2: Finding minimum beta* (Bisection Search) ---")
    
    # 1. Find a feasible upper bound
    beta_high = beta_init
    while not is_feasible(beta_high, Z_hat, D_obs, organ_sets_obs, params, epsilon_val, norm_settings):
        print(f"    Beta = {beta_high:.4f} is INFEASIBLE. Doubling beta_high.")
        beta_high *= 2.0
        if beta_high > 1e9:
            raise RuntimeError("Could not find a feasible beta_high. Model may be infeasible.")
    
    print(f"    Found feasible beta_high = {beta_high:.4f}.")
    
    # 2. Bisection search
    while beta_high - beta_low > gap_tol:
        beta_mid = 0.5 * (beta_low + beta_high)
        if is_feasible(beta_mid, Z_hat, D_obs, organ_sets_obs, params, epsilon_val, norm_settings):
            beta_high = beta_mid
        else:
            beta_low = beta_mid
            
    print(f"--- Bisection Search Complete. Final beta* = {beta_high:.4f} ---")
    return beta_high

# ----------------------------------------------------------------------
# 3. MAIN TRAINING FUNCTION
# ----------------------------------------------------------------------

def train_nonadd_smooth_inverse_model(n_obs):
    """Train non-additive smooth inverse optimization model for given training size."""
    print(f"\n{'='*60}")
    print(f"--- TRAINING NON-ADDITIVE SMOOTH MODEL FOR N = {n_obs} ---")
    print(f"{'='*60}\n")
    
    D_obs, w_obs, organ_sets_obs = load_data_nonadd_smooth(ML_DIR_NONADD_SMOOTH, n_obs)
    K_outcomes = 3
    
    Z_hat_obs = preprocess_outcomes_nonadd_smooth(w_obs, D_obs, organ_sets_obs, PARAMS_NONADD_SMOOTH)
    print(f"Computed Z_hat_obs. Shape: {Z_hat_obs.shape}")

    # --- STAGE 1: Minimize Epsilon ---
    print(f"\n--- STAGE 1: Minimizing Epsilon (at starting beta={BETA_0_NONADD_SMOOTH}) ---")
    
    vars_s1 = get_model_variables_nonadd(n_obs, K_outcomes, D_obs, organ_sets_obs)
    epsilon_s1_var = vars_s1["epsilon"]
    
    cons_s1 = build_constraints_nonadd_smooth(
        vars_s1, Z_hat_obs, D_obs, organ_sets_obs,
        PARAMS_NONADD_SMOOTH, beta_val=BETA_0_NONADD_SMOOTH, epsilon_val=epsilon_s1_var,
        norm_settings=NORMALIZATION_SETTINGS_NONADD_SMOOTH
    )
    
    prob_s1 = cp.Problem(cp.Minimize(epsilon_s1_var), cons_s1)
    prob_s1.solve(solver=SOLVER_NONADD_SMOOTH, verbose=SOLVER_VERBOSE_NONADD_SMOOTH)
    
    if prob_s1.status not in ("optimal", "optimal_inaccurate"):
        print(f"Stage 1 failed with status: {prob_s1.status}")
        return None
    
    epsilon_s1 = prob_s1.value
    epsilon_s2 = epsilon_s1 + EPSILON_TOL_NONADD_SMOOTH
    print(f"Stage 1 complete. epsilon* = {epsilon_s1:.6f}")
    
    # --- STAGE 2: Bisection Search for beta* ---
    beta_star = find_beta_bisect(
        Z_hat_obs, D_obs, organ_sets_obs, PARAMS_NONADD_SMOOTH,
        epsilon_val=epsilon_s2,
        beta_init=20,
        beta_low=BISECT_BETA_LOW,
        gap_tol=BISECT_GAP_TOL,
        norm_settings=NORMALIZATION_SETTINGS_NONADD_SMOOTH
    )
    
    # --- STAGE 3: Final solve with beta* ---
    print(f"\n--- STAGE 3: Maximize Delta (at beta*={beta_star:.4f}, eps={epsilon_s2:.8f}) ---")
    
    vars_s3 = get_model_variables_nonadd(n_obs, K_outcomes, D_obs, organ_sets_obs)
    
    cons_s3 = build_constraints_nonadd_smooth(
        vars_s3, Z_hat_obs, D_obs, organ_sets_obs,
        PARAMS_NONADD_SMOOTH, beta_val=beta_star, epsilon_val=epsilon_s2,
        norm_settings=NORMALIZATION_SETTINGS_NONADD_SMOOTH
    )
    
    prob_s3 = cp.Problem(cp.Maximize(cp.sum(vars_s3["delta_global"])), cons_s3)
    
    # MOSEK tolerances
    MOSEK_PARAMS = {
        'MSK_DPAR_INTPNT_CO_TOL_PFEAS': 1e-5,
        'MSK_DPAR_INTPNT_CO_TOL_DFEAS': 1e-5,
        'MSK_DPAR_INTPNT_CO_TOL_REL_GAP': 1e-5,
    }
    
    try:
        prob_s3.solve(solver=SOLVER_NONADD_SMOOTH, verbose=SOLVER_VERBOSE_NONADD_SMOOTH, mosek_params=MOSEK_PARAMS)
    except Exception:
        prob_s3.solve(solver=SOLVER_NONADD_SMOOTH, verbose=True)
    
    if prob_s3.status not in ("optimal", "optimal_inaccurate"):
        print(f"Stage 3 failed with final status: {prob_s3.status}")
        return None
    
    final_delta_global = vars_s3["delta_global"].value
    final_lambda = vars_s3["lambdas"].value
    
    print(f"\n--- Final Results for N={n_obs} ---")
    print(f"Final Global Epsilon*: {epsilon_s1:.8f}")
    print(f"Final Optimal Beta*: {beta_star:.4f}")
    print("\nFinal Delta Global (Cost f(z_i)):")
    print(np.round(final_delta_global, 4))
    print("\nFinal Lambdas (Gradient ∇f(z_i)):")
    print(np.round(final_lambda, 4))
    
    # Save Results
    print(f"\n--- Saving results to {RESULTS_DIR_NONADD_SMOOTH} ---")
    np.save(f'{RESULTS_DIR_NONADD_SMOOTH}/final_delta_global_N{n_obs}_smooth.npy', final_delta_global)
    np.save(f'{RESULTS_DIR_NONADD_SMOOTH}/final_lambda_N{n_obs}_smooth.npy', final_lambda)
    np.save(f'{RESULTS_DIR_NONADD_SMOOTH}/Z_hat_obs_N{n_obs}_smooth.npy', Z_hat_obs)
    np.save(f'{RESULTS_DIR_NONADD_SMOOTH}/beta_star_N{n_obs}_smooth.npy', np.array([beta_star]))
    np.save(f'{RESULTS_DIR_NONADD_SMOOTH}/epsilon_star_N{n_obs}_smooth.npy', np.array([epsilon_s1]))
    
    print(f"Successfully saved NON-ADDITIVE SMOOTH results for N={n_obs}.")

# Main training loop
if __name__ == "__main__":
    print("Starting NON-ADDITIVE SMOOTH model training...")
    for n_train in TRAINING_SIZES_NONADD_SMOOTH:
        try:
            train_nonadd_smooth_inverse_model(n_train)
        except Exception as e:
            print(f"ERROR training N={n_train}: {e}")
            traceback.print_exc()
    print("\nNON-ADDITIVE SMOOTH training complete!")

In [ ]:
#!/usr/bin/env python
# ============================================================================
# Non-Additive Non-Smooth Model - INVERSE OPTIMIZATION TRAINING
# ============================================================================
# This script implements the TWO-STAGE NON-ADDITIVE NON-SMOOTH IO model.
#
# CONCEPT:
# beta is fixed at a very large value (100,000) and not searched.
# This approximates the non-differentiable
# convex function behavior (the "max" function) within the smooth framework.
#
# STAGE 1: Minimize epsilon (suboptimality) using fixed BETA_NON_SMOOTH.
# STAGE 2: Maximize global delta (utility) using BETA_NON_SMOOTH and epsilon*.
# ============================================================================

import numpy as np
import cvxpy as cp
import time
import os
import traceback

# ----------------------------------------------------------------------
# 0. CONFIGURATION & CRITICAL PARAMETERS
# ----------------------------------------------------------------------

# Configuration
ML_DIR_NONADD_NONSMOOTH = "./ml_data_high_std"
RESULTS_DIR_NONADD_NONSMOOTH = "./results_high_std_non_add_nonsmooth"
os.makedirs(RESULTS_DIR_NONADD_NONSMOOTH, exist_ok=True)

TRAINING_SIZES = [20, 40, 60, 80, 100, 120, 140, 160, 180, 200] 

# Force beta to be large to approximate the non-smooth function.
BETA_NON_SMOOTH = 100000.0 
EPSILON_TOL = 1e-6 

NORMALIZATION_SETTINGS = {
    'idx_norm_zero': 0,
    'idx_norm_max': 1,
    'U_MAX': 10000.0
}

PARAMS = {
    'theta_prescribed': 78.0,
    'theta_max_PTV': 81.9,
    'theta_thresh_1': 59.15,
    'theta_thresh_2': 59.15,
    'beta_1': 0.5,
    'beta_2': 1.5
}

SOLVER = "MOSEK"
SOLVER_VERBOSE = False

# ----------------------------------------------------------------------
# 1. HELPER FUNCTIONS FOR DATA AND MODEL
# ----------------------------------------------------------------------

def load_data(data_dir, n_points):
    """Load training data."""
    try:
        D_all = np.load(f'{data_dir}/D_train.npy', allow_pickle=True)
        w_all = np.load(f'{data_dir}/w_train.npy', allow_pickle=True)
        organ_sets_all = np.load(f'{data_dir}/organ_sets_train.npy', allow_pickle=True)
        n_use = min(n_points, len(D_all))
        return D_all[:n_use], w_all[:n_use], organ_sets_all[:n_use]
    except Exception as e:
        print(f"Error loading data: {e}")
        raise

def preprocess_outcomes(w_obs, D_obs, organ_sets_obs, params):
    """Calculates Z_hat (OAR overdose sums and max dose)."""
    N = len(w_obs)
    K = 3
    Z_hat = np.zeros((N, K))
    for i in range(N):
        d_dose = D_obs[i] @ w_obs[i]
        org_i = organ_sets_obs[i]
        v_oar1 = org_i.get('oar1', np.array([]))
        v_oar2 = org_i.get('oar2', np.array([]))
        v_oar3 = org_i.get('oar3', np.array([]))
        
        if v_oar1.size > 0:
            overdose_1 = np.maximum(0, d_dose[v_oar1] - params['theta_thresh_1'])
            Z_hat[i, 0] = np.sum(overdose_1)
        if v_oar2.size > 0:
            overdose_2 = np.maximum(0, d_dose[v_oar2] - params['theta_thresh_2'])
            Z_hat[i, 1] = np.sum(overdose_2)
        if v_oar3.size > 0:
            Z_hat[i, 2] = np.max(d_dose[v_oar3])
    return Z_hat

def get_model_variables(N, K, D_obs, organ_sets_obs):
    """Create CVXPY variables for NON-ADDITIVE model."""
    delta_global = cp.Variable(N, nonneg=True, name="delta_global")
    lambdas = cp.Variable((N, K), name="lambda")
    epsilon = cp.Variable(nonneg=True, name="epsilon")
    
    # Dual variables (ragged dimensions)
    mu_1, mu_2, nu_1, nu_2, pi, sigma_1, sigma_2 = [], [], [], [], [], [], []
    for i in range(N):
        D_i = D_obs[i]
        org_i = organ_sets_obs[i]
        n_b = D_i.shape[1]
        
        mu_1.append(cp.Variable(len(org_i.get('ptv', [])), nonneg=True))
        mu_2.append(cp.Variable(len(org_i.get('ptv', [])), nonneg=True))
        nu_1.append(cp.Variable(len(org_i.get('oar1', [])), nonneg=True))
        nu_2.append(cp.Variable(len(org_i.get('oar2', [])), nonneg=True))
        pi.append(cp.Variable(len(org_i.get('oar3', [])), nonneg=True))
        sigma_1.append(cp.Variable(n_b, nonneg=True))
        sigma_2.append(cp.Variable(n_b, nonneg=True))
    
    return {
        "delta_global": delta_global, "lambdas": lambdas, "epsilon": epsilon,
        "mu_1": mu_1, "mu_2": mu_2, "nu_1": nu_1, "nu_2": nu_2,
        "pi": pi, "sigma_1": sigma_1, "sigma_2": sigma_2
    }

def build_constraints(variables, Z_hat, D_obs, organ_sets_obs, params,
                      beta_val, epsilon_val, norm_settings):
    """
    Build NON-ADDITIVE constraints with Vectorized C5.
    """
    delta_global = variables["delta_global"]
    lambdas = variables["lambdas"]
    N, K = Z_hat.shape
    cons = []
    
    print(f"--- Building constraints (beta={beta_val}) ---")
    
    if beta_val <= 0:
        raise ValueError("beta_val must be positive.")
        
    beta_inv = 1.0 / (2.0 * beta_val)

    # === C5: NON-ADDITIVE SMOOTH Convexity (Vectorized) ===
    # For large beta (small beta_inv), the quadratic term approaches 0,
    # approximating the subgradient inequality for non-smooth functions.
    
    # Create indices for all pairs where i != j
    idx_i_full = np.repeat(np.arange(N), N) 
    idx_j_full = np.tile(np.arange(N), N)   
    mask = idx_i_full != idx_j_full
    idx_i = idx_i_full[mask]
    idx_j = idx_j_full[mask]
    
    # 1. Linear Term: lambda_i^T * (Z_j - Z_i)
    Z_diff_mat = Z_hat[idx_j] - Z_hat[idx_i] 
    linear_term = cp.sum(cp.multiply(lambdas[idx_i], Z_diff_mat), axis=1)
    
    # 2. Quadratic Term: (1/2b) * ||lambda_j - lambda_i||^2
    lambda_diff = lambdas[idx_j] - lambdas[idx_i]
    # Row-wise sum of squares
    q_term = beta_inv * cp.sum(cp.square(lambda_diff), axis=1)
    
    # 3. Add Vectorized Constraint
    cons.append(
        delta_global[idx_j] - delta_global[idx_i] - linear_term >= q_term
    )
            
    # === C6: Monotonicity ===
    cons.append(lambdas >= 0)
    
    # === C7: Normalization ===
    idx_zero = norm_settings['idx_norm_zero']
    idx_max = norm_settings['idx_norm_max']
    u_val = norm_settings['U_MAX']
    
    if idx_zero < N: cons.append(delta_global[idx_zero] == 0)
    if idx_max < N:  cons.append(delta_global[idx_max] == u_val)
    # === C8: First-Order Sub-optimality ===
    for i in range(N):
        lam_i = lambdas[i, :]
        z_hat_i = Z_hat[i, :]
        D_i = D_obs[i]
        n_b = D_i.shape[1]
        
        # Dual variables
        mu_1_i = variables["mu_1"][i]
        mu_2_i = variables["mu_2"][i]
        nu_1_i = variables["nu_1"][i]
        nu_2_i = variables["nu_2"][i]
        pi_i   = variables["pi"][i]
        sigma_1_i = variables["sigma_1"][i]
        sigma_2_i = variables["sigma_2"][i]
        
        # h(lambda) - dual objective
        h_i = 0
        if mu_1_i.size > 0: h_i += cp.sum(mu_1_i) * params['theta_prescribed']
        if mu_2_i.size > 0: h_i -= cp.sum(mu_2_i) * params['theta_max_PTV']
        if nu_1_i.size > 0: h_i -= cp.sum(nu_1_i) * params['theta_thresh_1']
        if nu_2_i.size > 0: h_i -= cp.sum(nu_2_i) * params['theta_thresh_2']
        
        # C8
        cons.append(lam_i @ z_hat_i - h_i <= epsilon_val)
        
        # Dual feasibility
        if nu_1_i.size > 0: cons.append(nu_1_i <= lam_i[0])
        if nu_2_i.size > 0: cons.append(nu_2_i <= lam_i[1])
        if pi_i.size > 0:   cons.append(cp.sum(pi_i) <= lam_i[2])
            
        # Stationarity (D4)
        term_D_ptv = (mu_1_i - mu_2_i) @ D_i[organ_sets_obs[i].get('ptv',[]), :] if mu_1_i.size > 0 else 0
        term_D_oar1 = nu_1_i @ D_i[organ_sets_obs[i].get('oar1',[]), :] if nu_1_i.size > 0 else 0
        term_D_oar2 = nu_2_i @ D_i[organ_sets_obs[i].get('oar2',[]), :] if nu_2_i.size > 0 else 0
        term_D_oar3 = pi_i @ D_i[organ_sets_obs[i].get('oar3',[]), :] if pi_i.size > 0 else 0
            
        term_sigma_sum_1 = (params['beta_1'] / n_b) * cp.sum(sigma_1_i)
        term_sigma_sum_2 = (params['beta_2'] / n_b) * cp.sum(sigma_2_i)
        
        LHS_D4 = (-term_D_ptv + term_D_oar1 + term_D_oar2 + term_D_oar3
                  - sigma_1_i + term_sigma_sum_1
                  + sigma_2_i - term_sigma_sum_2)
        
        cons.append(LHS_D4 >= 0)
        
    return cons

# ----------------------------------------------------------------------
# 2. MAIN TRAINING FUNCTION
# ----------------------------------------------------------------------

def train_nonadd_nonsmooth_inverse_model(n_obs):
    """Train non-additive non-smooth inverse optimization model for given training size."""
    print(f"\n{'='*60}")
    print(f"--- TRAINING NON-ADDITIVE NON-SMOOTH MODEL FOR N = {n_obs} ---")
    print(f"{'='*60}\n")
    
    # 1. Load Data
    D_obs, w_obs, organ_sets_obs = load_data(ML_DIR_NONADD_NONSMOOTH, n_obs)
    K_outcomes = 3
    Z_hat_obs = preprocess_outcomes(w_obs, D_obs, organ_sets_obs, PARAMS)
    
    # --- STAGE 1: Minimize Epsilon with Fixed High Beta ---
    print(f"\n--- STAGE 1: Minimizing Epsilon (Fixed Beta={BETA_NON_SMOOTH}) ---")
    
    vars_s1 = get_model_variables(n_obs, K_outcomes, D_obs, organ_sets_obs)
    epsilon_s1_var = vars_s1["epsilon"]
    
    cons_s1 = build_constraints(
        vars_s1, Z_hat_obs, D_obs, organ_sets_obs,
        PARAMS, beta_val=BETA_NON_SMOOTH, epsilon_val=epsilon_s1_var,
        norm_settings=NORMALIZATION_SETTINGS
    )
    
    prob_s1 = cp.Problem(cp.Minimize(epsilon_s1_var), cons_s1)
    
    # MOSEK tolerances (Stage 1)
    MOSEK_PARAMS_S1 = {
        'MSK_DPAR_INTPNT_CO_TOL_PFEAS': 1e-6,
        'MSK_DPAR_INTPNT_CO_TOL_DFEAS': 1e-6,
        'MSK_DPAR_INTPNT_CO_TOL_REL_GAP': 1e-6,
    }
    
    prob_s1.solve(solver=SOLVER, verbose=SOLVER_VERBOSE, mosek_params=MOSEK_PARAMS_S1)
    
    if prob_s1.status not in ("optimal", "optimal_inaccurate"):
        print(f"Stage 1 failed with status: {prob_s1.status}")
        return None
    
    epsilon_s1 = prob_s1.value
    epsilon_s2 = epsilon_s1 + EPSILON_TOL
    print(f"Stage 1 complete. epsilon* = {epsilon_s1:.6f}")
    
    # --- STAGE 2: Maximize Delta with Fixed High Beta ---
    # (Note: We skip Bisection Search because we are enforcing Non-Smoothness via large Beta)
    print(f"\n--- STAGE 2: Maximize Delta (Fixed Beta={BETA_NON_SMOOTH}, eps={epsilon_s2:.8f}) ---")
    
    vars_s3 = get_model_variables(n_obs, K_outcomes, D_obs, organ_sets_obs)
    
    cons_s3 = build_constraints(
        vars_s3, Z_hat_obs, D_obs, organ_sets_obs,
        PARAMS, beta_val=BETA_NON_SMOOTH, epsilon_val=epsilon_s2,
        norm_settings=NORMALIZATION_SETTINGS
    )
    
    prob_s3 = cp.Problem(cp.Maximize(cp.sum(vars_s3["delta_global"])), cons_s3)
    
    # MOSEK tolerances (Stage 2)
    MOSEK_PARAMS_S2 = {
        'MSK_DPAR_INTPNT_CO_TOL_PFEAS': 1e-5,
        'MSK_DPAR_INTPNT_CO_TOL_DFEAS': 1e-5,
        'MSK_DPAR_INTPNT_CO_TOL_REL_GAP': 1e-5,
    }
    
    prob_s3.solve(solver=SOLVER, verbose=SOLVER_VERBOSE, mosek_params=MOSEK_PARAMS_S2)
    
    if prob_s3.status not in ("optimal", "optimal_inaccurate"):
        print(f"Stage 2 failed with status: {prob_s3.status}")
        return None
    
    final_delta_global = vars_s3["delta_global"].value
    final_lambda = vars_s3["lambdas"].value
    
    print(f"\n--- Final Results for N={n_obs} ---")
    print(f"Final Global Epsilon*: {epsilon_s1:.8f}")
    print(f"Final Forced Beta*: {BETA_NON_SMOOTH:.1f}")
    print("\nFinal Delta Global (Cost f(z_i)):")
    print(np.round(final_delta_global, 4))
    print("\nFinal Lambdas (Gradient ∇f(z_i)):")
    print(np.round(final_lambda, 4))
    
    # Save Results (Suffix vNS for Non-Smooth)
    print(f"\n--- Saving results to {RESULTS_DIR_NONADD_NONSMOOTH} ---")
    np.save(f'{RESULTS_DIR_NONADD_NONSMOOTH}/final_delta_global_N{n_obs}_convex.npy', final_delta_global)
    np.save(f'{RESULTS_DIR_NONADD_NONSMOOTH}/final_lambda_N{n_obs}_convex.npy', final_lambda)
    np.save(f'{RESULTS_DIR_NONADD_NONSMOOTH}/Z_hat_obs_N{n_obs}_convex.npy', Z_hat_obs)
    # We save the forced beta as beta_star so evaluation scripts can pick it up
    np.save(f'{RESULTS_DIR_NONADD_NONSMOOTH}/beta_star_N{n_obs}_convex.npy', np.array([BETA_NON_SMOOTH]))
    np.save(f'{RESULTS_DIR_NONADD_NONSMOOTH}/epsilon_star_N{n_obs}_convex.npy', np.array([epsilon_s1]))
    
    print(f"Successfully saved NON-ADDITIVE NON-SMOOTH results for N={n_obs}.")

# Main training loop
if __name__ == "__main__":
    print("Starting NON-ADDITIVE NON-SMOOTH model training...")
    for n_train in TRAINING_SIZES:
        try:
            train_nonadd_nonsmooth_inverse_model(n_train)
        except Exception as e:
            print(f"ERROR training N={n_train}: {e}")
            traceback.print_exc()
    print("\nNON-ADDITIVE NON-SMOOTH training complete!")